# 02 · Neural Networks from Scratch (NumPy)

**Part I — Foundations** · Estimated time: ~120 min

In this notebook we build multi-layer perceptrons with nothing but NumPy. There is no autograd, no
`nn.Module`, no optimizer library: every forward pass is a handful of matrix products that we write
ourselves, and every gradient is derived on paper, implemented by hand and verified numerically.
By the end you will have trained a network that reads handwritten digits with roughly 97% accuracy,
and — more importantly — you will know exactly what every line of that network does. When we switch to
PyTorch in Notebook 03 · Tensors and Autograd, nothing will feel like magic.

## Learning objectives

- Explain what an artificial neuron computes and why a single neuron cannot solve XOR.
- Describe the common activation functions, their derivatives and their failure modes (saturation, dead ReLUs).
- Write the forward pass of a multi-layer perceptron with explicit shapes and count its parameters.
- Derive backpropagation for linear layers, ReLU and softmax cross-entropy using the chain rule, with every shape checked.
- Verify hand-written gradients against finite differences with a gradient checker.
- Implement a small modular library (`Linear`, `ReLU`, `Sequential`, `SGD`) and a mini-batch training loop.
- Train an MLP on MNIST in pure NumPy and analyse its errors with a confusion matrix.
- Explain the universal approximation theorem in practice and why hand-written backprop stops scaling.

## Prerequisites

- Notebook 00 · Course Overview and Setup (environment, the `FAST_RUN` convention, the `../data` folder)
- Notebook 01 · Mathematical Foundations (matrix products and shapes, the chain rule in matrix form, softmax and log-sum-exp, gradient descent, the gradient checker)

## Contents

1. From perceptron to MLP
2. Activation functions
3. The forward pass with explicit shapes
4. Loss functions
5. Backpropagation, derived and checked
6. A modular implementation
7. MNIST in NumPy
8. Universal approximation in practice
- Exercises · Summary · Further reading

In [ ]:
import math, os, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.special import erf
from sklearn.datasets import make_moons

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# In this notebook PyTorch is used for exactly one thing: downloading MNIST (Section 7).
# Every network, gradient and optimizer below is written in NumPy.
print(f"NumPy {np.__version__} | FAST_RUN = {FAST_RUN}")

## 1. From perceptron to MLP

### 1.1 The artificial neuron

The basic unit of every network in this course is an **artificial neuron**. It receives a vector of
inputs $\mathbf{x} \in \mathbb{R}^d$, forms a weighted sum with its **weights** $\mathbf{w} \in \mathbb{R}^d$
and a **bias** $b \in \mathbb{R}$, and passes the result through an **activation function** $\phi$:

$$
z = \mathbf{w}^\top \mathbf{x} + b, \qquad a = \phi(z).
$$

The quantity $z$ is called the **pre-activation**, $a$ the **activation**. The bias shifts the threshold
at which the neuron "fires"; without it every decision boundary would be forced through the origin.

The oldest choice for $\phi$ is the Heaviside step, $\phi(z) = \mathbb{1}[z > 0]$, which gives Rosenblatt's
1958 **perceptron**: the neuron outputs 1 when the weighted evidence exceeds the threshold $-b$ and 0
otherwise. Geometrically, the set of inputs with $z = 0$ is a hyperplane (a line in two dimensions), and the
perceptron labels the two half-spaces on either side. A perceptron is therefore a **linear classifier**:
it can only separate classes that can be split by a single straight line (or plane).

Let us make this concrete with the smallest possible datasets: logic gates on two binary inputs.

In [ ]:
def step(z):
    """Heaviside step function, the perceptron's activation."""
    return (z > 0).astype(float)


def perceptron_predict(X, w, b):
    """Apply one perceptron to a batch of inputs X of shape (N, d)."""
    return step(X @ w + b)


# The four corners of the unit square are the inputs of every two-input logic gate.
X_gate = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1], dtype=float)
y_or = np.array([0, 1, 1, 1], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

# A hand-designed AND neuron: it fires only when both inputs are on (1 + 1 - 1.5 > 0).
w_and, b_and = np.array([1.0, 1.0]), -1.5
print("inputs:\n", X_gate)
print("AND by hand :", perceptron_predict(X_gate, w_and, b_and))
print("AND target  :", y_and)

### 1.2 Learning the weights: the perceptron algorithm

Designing weights by hand does not scale, so Rosenblatt also proposed a learning rule. Present the
examples one by one; whenever the prediction $\hat{y}$ is wrong, nudge the weights towards the
misclassified input:

$$
\mathbf{w} \leftarrow \mathbf{w} + \eta\,(y - \hat{y})\,\mathbf{x}, \qquad b \leftarrow b + \eta\,(y - \hat{y}),
$$

where $\eta > 0$ is the **learning rate**. If the target was 1 but we predicted 0, the update adds a multiple of
$\mathbf{x}$ to $\mathbf{w}$, which increases $\mathbf{w}^\top \mathbf{x}$ and pushes the example to the correct
side; the opposite happens for the other kind of mistake. The **perceptron convergence theorem** guarantees
that this procedure stops after finitely many updates *if* the data is linearly separable. What happens
when it is not is the subject of the next experiment.

We also write a small plotting helper that colours the plane according to a classifier's prediction; we
will reuse it for every 2-D experiment in this notebook.

In [ ]:
def plot_decision_boundary(predict, X, y, ax=None, title="", padding=0.5, resolution=200):
    """Colour the plane by the label that `predict` assigns to each point.

    predict: function mapping an array of shape (M, 2) to M integer labels.
    """
    if ax is None:
        _, ax = plt.subplots()
    x_min, x_max = X[:, 0].min() - padding, X[:, 0].max() + padding
    y_min, y_max = X[:, 1].min() - padding, X[:, 1].max() + padding
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution), np.linspace(y_min, y_max, resolution))
    grid = np.stack([xx.ravel(), yy.ravel()], axis=1).astype(X.dtype)
    zz = np.asarray(predict(grid)).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=40)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_title(title)
    return ax


def train_perceptron(X, y, lr=0.1, epochs=50, seed=SEED):
    """Rosenblatt's perceptron learning rule. Returns weights, bias and the mistakes per epoch."""
    rng = np.random.default_rng(seed)
    w, b = rng.normal(0.0, 0.1, size=X.shape[1]), 0.0
    mistakes = []
    for _ in range(epochs):
        n_errors = 0
        for x_i, y_i in zip(X, y):
            y_hat = step(x_i @ w + b)
            if y_hat != y_i:
                w += lr * (y_i - y_hat) * x_i
                b += lr * (y_i - y_hat)
                n_errors += 1
        mistakes.append(n_errors)
        if n_errors == 0:          # every example is correct: the algorithm has converged
            break
    return w, b, mistakes


fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, y_gate) in zip(axes, [("AND", y_and), ("OR", y_or), ("XOR", y_xor)]):
    w, b, mistakes = train_perceptron(X_gate, y_gate)
    acc = np.mean(perceptron_predict(X_gate, w, b) == y_gate)
    print(f"{name}: stopped after {len(mistakes):2d} epochs, mistakes in last epoch = {mistakes[-1]}, "
          f"accuracy = {acc:.2f}")
    plot_decision_boundary(lambda G, w=w, b=b: perceptron_predict(G, w, b), X_gate, y_gate, ax=ax,
                           title=f"perceptron on {name} (acc {acc:.2f})")
plt.tight_layout(); plt.show()

The perceptron learns AND and OR in a few epochs: both gates are linearly separable, so a single line
suffices. On **XOR** the algorithm runs for the full 50 epochs and keeps making mistakes. No line can put
$(0,1)$ and $(1,0)$ on one side and $(0,0)$ and $(1,1)$ on the other, so no setting of $\mathbf{w}, b$ can
ever be correct, and the learning rule keeps oscillating between different wrong answers. This tiny
observation — published by Minsky and Papert in 1969 — was one reason research on neural networks stalled
for more than a decade.

### 1.3 Hidden layers as learned features

The way out is to let one neuron use the outputs of *other* neurons as its inputs. Consider two
perceptrons that compute OR and AND of the inputs, followed by a third one that computes "OR and not AND":

$$
h_1 = \mathbb{1}[x_1 + x_2 - 0.5 > 0], \qquad
h_2 = \mathbb{1}[x_1 + x_2 - 1.5 > 0], \qquad
\hat{y} = \mathbb{1}[h_1 - h_2 - 0.5 > 0].
$$

The intermediate values $h_1, h_2$ form a **hidden layer**. Their job is to re-describe the input in a new
coordinate system in which the problem *is* linearly separable. This is the essential idea behind every
deep network: stacks of layers learn a sequence of representations, each of which makes the next
layer's job easier. The final layer is still a linear classifier — it just operates on learned features
rather than on raw pixels.

In [ ]:
def xor_two_layer(X):
    """A hand-designed two-layer network for XOR. Returns predictions and the hidden features."""
    W1 = np.array([[1.0, 1.0],
                   [1.0, 1.0]])            # shape (2 inputs, 2 hidden units)
    b1 = np.array([-0.5, -1.5])           # unit 1 computes OR, unit 2 computes AND
    W2 = np.array([1.0, -1.0])            # output = OR - AND
    b2 = -0.5
    H = step(X @ W1 + b1)                 # (N, 2) hidden features
    return step(H @ W2 + b2), H


y_hat, H = xor_two_layer(X_gate)
print("XOR prediction:", y_hat, "| target:", y_xor)
print("hidden features (OR, AND) for each input:\n", H)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_decision_boundary(lambda G: xor_two_layer(G)[0], X_gate, y_xor, ax=axes[0],
                       title="input space: XOR needs two lines")
axes[1].scatter(H[:, 0] + 0.02 * np.arange(4), H[:, 1], c=y_xor, cmap="coolwarm", edgecolors="k", s=80)
axes[1].plot([-0.2, 1.2], [-0.7, 0.7], "k--", label="$h_1 - h_2 = 0.5$")
axes[1].set_xlabel("$h_1$ = OR"); axes[1].set_ylabel("$h_2$ = AND")
axes[1].set_title("hidden space: one line separates the classes"); axes[1].legend()
plt.tight_layout(); plt.show()

In the input space (left) the positive class needs a band between two parallel lines — the two hidden
units. In the hidden space (right) the two positive inputs $(0,1)$ and $(1,0)$ collapse onto the same
point $(h_1, h_2) = (1, 0)$, and a single line separates it from $(0,0)$ and $(1,1)$. The network has
built a feature ("exactly one input is on") that makes the problem trivial.

A network of this shape — an input layer, one or more hidden layers, an output layer, with every unit of
one layer connected to every unit of the next — is a **multi-layer perceptron (MLP)**, also called a
fully-connected or dense feed-forward network. Two things stand between the hand-designed XOR network
and a network that *learns*: the step function has a zero derivative everywhere except at the jump (where
it has none), so no gradient-based learning rule can be applied to it, and we have not yet said how to move
gradients through several layers.
The first problem is solved by smooth activation functions (Section 2), the second by backpropagation
(Section 5).

> **Key idea.** A single neuron is a linear classifier. Hidden layers transform the input into a new
> representation in which the classes become linearly separable; learning those representations from data
> is what deep learning is about.

## 2. Activation functions

### 2.1 Why we need non-linearity

Suppose we stack two *linear* layers without any activation in between: $\mathbf{h} = W_1 \mathbf{x} + \mathbf{b}_1$
and $\mathbf{y} = W_2 \mathbf{h} + \mathbf{b}_2$. Substituting the first into the second gives

$$
\mathbf{y} = W_2 (W_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = \underbrace{(W_2 W_1)}_{W_{\text{eff}}} \mathbf{x} + \underbrace{(W_2 \mathbf{b}_1 + \mathbf{b}_2)}_{\mathbf{b}_{\text{eff}}},
$$

which is again a single linear layer. By induction, any number of stacked linear layers collapses to one:
depth buys nothing. The hidden layer only becomes useful when a **non-linear** function $\phi$ is applied
element-wise between the layers. (The step function in Section 1 was non-linear, but not differentiable;
we now want both.) Let us check the collapse numerically before looking at the activations we will use. In
the code the batch sits on the left of every product, `X @ W1`, so the collapsed weight is `W1 @ W2` — the
same product as $W_2 W_1$ above, written for row vectors; Section 3 explains this convention.

In [ ]:
rng = np.random.default_rng(SEED)
W1, b1 = rng.normal(size=(3, 4)), rng.normal(size=4)
W2, b2 = rng.normal(size=(4, 2)), rng.normal(size=2)
X = rng.normal(size=(5, 3))                      # a batch of 5 inputs with 3 features

two_linear_layers = (X @ W1 + b1) @ W2 + b2      # shape (5, 2)
W_eff, b_eff = W1 @ W2, b1 @ W2 + b2             # shapes (3, 2) and (2,)
one_linear_layer = X @ W_eff + b_eff

print("two stacked linear layers == one linear layer:", np.allclose(two_linear_layers, one_linear_layer))

### 2.2 The classical and the modern activations

We will meet five activation functions in this course. Each is applied element-wise to the pre-activation
$z$, and for backpropagation we need its derivative.

| name | definition | derivative | range |
|---|---|---|---|
| sigmoid | $\sigma(z) = \dfrac{1}{1 + e^{-z}}$ | $\sigma(z)\,(1 - \sigma(z))$ | $(0, 1)$ |
| tanh | $\tanh(z) = \dfrac{e^{z} - e^{-z}}{e^{z} + e^{-z}}$ | $1 - \tanh^2(z)$ | $(-1, 1)$ |
| ReLU | $\max(0, z)$ | $\mathbb{1}[z > 0]$ | $[0, \infty)$ |
| leaky ReLU | $\max(\alpha z, z)$, small $\alpha$ | $1$ if $z>0$ else $\alpha$ | $(-\infty, \infty)$ |
| GELU | $z\,\Phi(z)$, $\Phi$ the standard normal CDF | $\Phi(z) + z\,\varphi(z)$ | $\approx[-0.17, \infty)$ |

A few remarks on the derivatives. For the sigmoid, differentiate $\sigma(z) = (1 + e^{-z})^{-1}$ with the
chain rule: $\sigma'(z) = e^{-z} (1 + e^{-z})^{-2} = \sigma(z) \cdot \frac{e^{-z}}{1 + e^{-z}} = \sigma(z)(1 - \sigma(z))$.
The derivative is expressed through the function's own output, which is convenient in code: once the
forward pass has computed $\sigma(z)$, the backward pass needs no extra exponentials. The same is true
for tanh. The ReLU derivative is a step function; at exactly $z = 0$ the derivative does not exist, and by
convention we use 0 there (the event has probability zero for continuous inputs). GELU (Hendrycks and
Gimpel, 2016) is a smooth relative of ReLU that multiplies the input by the probability that a standard
normal variable is below it; it is the default in Transformers (Notebook 09 · Attention and Transformers).

A numerically careful sigmoid deserves a comment. The naive formula evaluates $e^{-z}$, which overflows
in float64 for $z < -709$ and in float32 far earlier. The standard trick is to compute $e^{-|z|}$, which is
always in $(0, 1]$, and to use the two algebraically equal forms $\sigma(z) = \frac{1}{1+e^{-z}}$ for $z \ge 0$
and $\sigma(z) = \frac{e^{z}}{1+e^{z}}$ for $z < 0$.

In [ ]:
def sigmoid(z):
    """Numerically stable logistic sigmoid: never exponentiates a positive number."""
    e = np.exp(-np.abs(z))
    return np.where(z >= 0, 1.0 / (1.0 + e), e / (1.0 + e))


def d_sigmoid(z):
    s = sigmoid(z)
    return s * (1.0 - s)


def tanh(z):
    return np.tanh(z)


def d_tanh(z):
    return 1.0 - np.tanh(z) ** 2


def relu(z):
    return np.maximum(0.0, z)


def d_relu(z):
    return (z > 0).astype(z.dtype)


def leaky_relu(z, alpha=0.1):
    return np.where(z > 0, z, alpha * z)


def d_leaky_relu(z, alpha=0.1):
    return np.where(z > 0, 1.0, alpha)


def gelu(z):
    Phi = 0.5 * (1.0 + erf(z / np.sqrt(2.0)))          # standard normal CDF
    return z * Phi


def d_gelu(z):
    Phi = 0.5 * (1.0 + erf(z / np.sqrt(2.0)))
    phi = np.exp(-0.5 * z ** 2) / np.sqrt(2.0 * np.pi)  # standard normal density
    return Phi + z * phi


activations = {
    "sigmoid": (sigmoid, d_sigmoid), "tanh": (tanh, d_tanh), "ReLU": (relu, d_relu),
    "leaky ReLU": (leaky_relu, d_leaky_relu), "GELU": (gelu, d_gelu),
}
z = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, (f, df) in activations.items():
    axes[0].plot(z, f(z), label=name)
    axes[1].plot(z, df(z), label=name)
axes[0].set_title("activation functions"); axes[0].set_xlabel("$z$"); axes[0].set_ylabel("$\\phi(z)$")
axes[0].set_ylim(-1.5, 3); axes[0].legend()
axes[1].set_title("their derivatives"); axes[1].set_xlabel("$z$"); axes[1].set_ylabel("$\\phi'(z)$")
axes[1].set_ylim(-0.2, 1.3); axes[1].legend()
plt.tight_layout(); plt.show()

# sanity check of the hand-derived derivatives against central finite differences
eps = 1e-6
for name, (f, df) in activations.items():
    numeric = (f(z + eps) - f(z - eps)) / (2 * eps)
    print(f"{name:11s}: max |analytic - numeric| = {np.max(np.abs(df(z) - numeric)):.2e}")

The finite-difference check confirms every derivative to within about $10^{-9}$ (for the ReLU family the
check could only fail at a grid point within $\varepsilon$ of the kink at 0, and none of our 400 points is
that close). Two features of the right-hand plot matter enormously for training.

**Saturation.** The derivatives of sigmoid and tanh are bell-shaped and vanish as $|z|$ grows: for
$|z| > 5$ the sigmoid's slope is below $0.007$. A neuron whose pre-activations are large is **saturated**;
almost no gradient flows through it, and since gradients are multiplied layer by layer during
backpropagation (Section 5), a few saturated layers in a row can shrink the gradient to nothing. This
**vanishing gradient** problem is why deep sigmoid networks were so hard to train, and it is revisited in
Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization. The
sigmoid has the additional disadvantage that its maximum slope is
only $0.25$, so even unsaturated sigmoid layers attenuate the gradient by a factor of at least four.

**Dead ReLUs.** The ReLU derivative is exactly 1 for positive inputs — gradients pass through unchanged, no
matter how deep the network — and exactly 0 for negative inputs. If a unit's pre-activation is negative for
*every* input in the dataset, the unit outputs 0 everywhere, receives zero gradient, and can never recover:
it is **dead**. This typically happens after an over-large update pushes a bias or a weight vector far
into the negative region. Leaky ReLU and GELU keep a small non-zero slope on the negative side precisely
to avoid this. The next cell simulates the phenomenon by shifting the biases of a random layer.

In [ ]:
rng = np.random.default_rng(SEED)
X = rng.normal(size=(1000, 50))                         # 1000 inputs with 50 features
W = rng.normal(0.0, np.sqrt(2.0 / 50), size=(50, 100))  # a layer with 100 ReLU units

print("sigmoid slope at z = 0, 2, 5, 10:", np.round(d_sigmoid(np.array([0.0, 2.0, 5.0, 10.0])), 5))
for shift in [0.0, -2.0, -4.0, -6.0]:
    A = relu(X @ W + shift)
    dead_fraction = np.mean(np.all(A == 0, axis=0))     # units that are zero for every input
    print(f"bias shifted by {shift:+.0f}: {dead_fraction:5.1%} of the units are dead")

With unbiased pre-activations, no unit is dead — each is active for roughly half the inputs. Shifting all
biases to $-4$ kills about a fifth of the units, and at $-6$ essentially the whole layer is silent. Nothing
in gradient descent can bring a dead unit back, so in practice we guard against this with sensible
initialization and learning rates (Section 7 and Notebook 04).

> **Try it.** Change `alpha` in `leaky_relu` to 0.01 and 0.3 and re-plot. Which value would you choose,
> and why might a very large `alpha` defeat the purpose of the non-linearity?

## 3. The forward pass with explicit shapes

### 3.1 Batches, weights and the affine layer

Networks process many examples at once. We stack $N$ input vectors as the rows of a matrix
$X \in \mathbb{R}^{N \times d}$ — the convention **(batch, features)** used throughout the course. A
**linear layer** (also called affine or fully-connected) with $d_{\text{in}}$ inputs and $d_{\text{out}}$ outputs
has a weight matrix $W \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$ and a bias $\mathbf{b} \in \mathbb{R}^{d_{\text{out}}}$,
and maps the whole batch in one matrix product:

$$
Z = X W + \mathbf{1}\,\mathbf{b}^\top, \qquad
\underbrace{Z}_{N \times d_{\text{out}}} = \underbrace{X}_{N \times d_{\text{in}}}\; \underbrace{W}_{d_{\text{in}} \times d_{\text{out}}} + \underbrace{\mathbf{1}}_{N \times 1}\;\underbrace{\mathbf{b}^\top}_{1 \times d_{\text{out}}}.
$$

Row $n$ of $Z$ is $\mathbf{x}_n^\top W + \mathbf{b}^\top$, i.e. the pre-activations of all $d_{\text{out}}$ units for
example $n$; column $j$ of $W$ holds the weights of unit $j$. The outer product $\mathbf{1}\mathbf{b}^\top$
simply copies the bias into every row — in NumPy, `X @ W + b` does this automatically by broadcasting
(Notebook 01 · Mathematical Foundations, Section 5). We store $W$ as (in, out) rather than (out, in) so that
the batch stays on the left and shapes read naturally from left to right; PyTorch's `nn.Linear` stores
the transpose, which changes nothing mathematically.

### 3.2 A two-hidden-layer network

A network with layer sizes $d_0 = d$ (inputs), $d_1, d_2$ (hidden) and $d_3 = k$ (outputs) computes

$$
\begin{aligned}
Z_1 &= X W_1 + \mathbf{1}\mathbf{b}_1^\top, & A_1 &= \phi(Z_1), && (N \times d_1) \\
Z_2 &= A_1 W_2 + \mathbf{1}\mathbf{b}_2^\top, & A_2 &= \phi(Z_2), && (N \times d_2) \\
Z_3 &= A_2 W_3 + \mathbf{1}\mathbf{b}_3^\top. & && && (N \times k)
\end{aligned}
$$

The final pre-activations $Z_3$ are the **logits**: one real number per class, later turned into
probabilities by the softmax. There is no activation after the last layer because the loss function
(Section 4) takes care of that. In the implementation we keep every intermediate quantity in a `cache`,
because backpropagation will need $X, Z_1, A_1, Z_2, A_2$ again.

In [ ]:
def init_mlp(sizes, rng):
    """He-initialised parameters for an MLP with the given layer sizes, e.g. [784, 256, 128, 10]."""
    params = {}
    for l, (fan_in, fan_out) in enumerate(zip(sizes[:-1], sizes[1:]), start=1):
        params[f"W{l}"] = rng.normal(0.0, np.sqrt(2.0 / fan_in), size=(fan_in, fan_out))
        params[f"b{l}"] = np.zeros(fan_out)
    return params


def mlp_forward(params, X):
    """Forward pass of a 2-hidden-layer ReLU MLP. Returns the logits and a cache for backprop."""
    W1, b1, W2, b2, W3, b3 = (params[k] for k in ["W1", "b1", "W2", "b2", "W3", "b3"])
    Z1 = X @ W1 + b1          # (N, d1)   pre-activations of hidden layer 1
    A1 = relu(Z1)             # (N, d1)   activations of hidden layer 1
    Z2 = A1 @ W2 + b2         # (N, d2)
    A2 = relu(Z2)             # (N, d2)
    Z3 = A2 @ W3 + b3         # (N, k)    logits
    cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return Z3, cache


rng = np.random.default_rng(SEED)
sizes = [4, 8, 6, 3]                       # 4 inputs, hidden layers of 8 and 6 units, 3 classes
params = init_mlp(sizes, rng)
X_demo = rng.normal(size=(5, 4))           # a batch of N = 5 examples
logits, cache = mlp_forward(params, X_demo)

for name, arr in params.items():
    print(f"{name}: {str(arr.shape):10s}", end="   ")
print()
for name, arr in cache.items():
    print(f"{name}: {str(arr.shape):10s}", end="   ")
print(f"logits: {logits.shape}")

The shapes chain together exactly as the equations say: $(5, 4)\,(4, 8) \to (5, 8)$, then $(5, 8)\,(8, 6) \to (5, 6)$,
then $(5, 6)\,(6, 3) \to (5, 3)$. Whenever you write a network by hand, print the shapes once; almost every
bug in this notebook announces itself as a shape mismatch.

The weights are drawn from $\mathcal{N}(0, 2/d_{\text{in}})$, the **He initialization** (He et al., 2015). We
justify the factor $2/d_{\text{in}}$ properly in Notebook 04 · Training Deep Networks: Optimization,
Initialization, Normalization, Regularization; for now, note that it keeps the scale of the
activations roughly constant from layer to layer when the activation is ReLU, and that Section 7 shows
experimentally what goes wrong with other choices.

### 3.3 Counting parameters

A linear layer from $d_{\text{in}}$ to $d_{\text{out}}$ units has $d_{\text{in}} d_{\text{out}}$ weights and $d_{\text{out}}$ biases, so
an MLP with sizes $d_0, d_1, \dots, d_L$ has

$$
\#\text{params} = \sum_{l=1}^{L} (d_{l-1} + 1)\, d_l .
$$

The MNIST network of Section 7 has sizes $784, 256, 128, 10$ and therefore
$785 \cdot 256 + 257 \cdot 128 + 129 \cdot 10 = 200{,}960 + 32{,}896 + 1{,}290 = 235{,}146$ parameters. The
first layer dominates: it alone holds 85% of the parameters because it touches every pixel. This is a
first hint of why convolutional networks (Notebook 06 · Convolutional Neural Networks) share weights across
positions instead.

In [ ]:
def count_params(params):
    return sum(p.size for p in params.values())


for sizes in [[4, 8, 6, 3], [784, 256, 128, 10], [784, 1024, 1024, 10]]:
    n_formula = sum((d_prev + 1) * d_next for d_prev, d_next in zip(sizes[:-1], sizes[1:]))
    n_counted = count_params(init_mlp(sizes, rng))
    print(f"sizes {sizes}: formula = {n_formula:,}   counted = {n_counted:,}")

## 4. Loss functions

A network with random weights produces random outputs. To *train* it we need a single number that says
how bad the outputs are — the **loss** — and that number must be a differentiable function of the
parameters, so that we can lower it by gradient descent. The loss encodes the task: regression and
classification use different losses, and Notebook 01 · Mathematical Foundations derived both from maximum
likelihood. Here we recall
the results and implement them carefully.

### 4.1 Mean squared error for regression

For a regression problem with targets $Y \in \mathbb{R}^{N \times k}$ and predictions $\hat{Y}$ we use the
**mean squared error**

$$
L_{\text{MSE}} = \frac{1}{2Nk} \sum_{n=1}^{N} \sum_{j=1}^{k} (\hat{y}_{nj} - y_{nj})^2,
\qquad
\frac{\partial L_{\text{MSE}}}{\partial \hat{Y}} = \frac{1}{Nk}\,(\hat{Y} - Y).
$$

The factor $\frac12$ is a convention that cancels the 2 from differentiating the square; it changes the
loss by a constant factor and nothing else. Averaging (rather than summing) over the batch keeps the loss
and the gradient on the same scale whatever the batch size, which lets us reuse one learning rate for
different batch sizes.

### 4.2 Softmax cross-entropy for classification

For classification into $k$ classes the network produces logits $\mathbf{z} \in \mathbb{R}^k$ per example, which
the **softmax** turns into a probability vector,

$$
p_j = \operatorname{softmax}(\mathbf{z})_j = \frac{e^{z_j}}{\sum_{i=1}^{k} e^{z_i}}, \qquad p_j > 0,\; \sum_j p_j = 1.
$$

If the true class of example $n$ is $y_n$, the **cross-entropy loss** is the average negative
log-probability that the model assigns to the correct class:

$$
L_{\text{CE}} = -\frac{1}{N} \sum_{n=1}^{N} \log p_{n, y_n}
= \frac{1}{N} \sum_{n=1}^{N} \Big( \log \sum_{i} e^{z_{ni}} - z_{n, y_n} \Big).
$$

Notebook 01 showed that this is exactly the negative log-likelihood of a categorical model, and that
computing it as written on the right — via the **log-sum-exp** — is essential for numerical stability.
Exponentiating a logit of 1000 overflows float64; subtracting the row-wise maximum first,
$\log \sum_i e^{z_i} = m + \log \sum_i e^{z_i - m}$ with $m = \max_i z_i$, gives the same value with every
exponent at most 0. We implement `log_softmax` this way and derive both `softmax` and the loss from it.

In [ ]:
def mse_loss(Y_hat, Y):
    """Mean squared error with the 1/2 convention, averaged over every output element."""
    return 0.5 * np.mean((Y_hat - Y) ** 2)


def log_softmax(Z):
    """Row-wise log-softmax of logits Z of shape (N, k), computed with the log-sum-exp trick."""
    Z_shifted = Z - Z.max(axis=1, keepdims=True)                    # every entry now <= 0
    return Z_shifted - np.log(np.sum(np.exp(Z_shifted), axis=1, keepdims=True))


def softmax(Z):
    return np.exp(log_softmax(Z))


def cross_entropy(logits, y):
    """Average negative log-probability of the true classes y (integer labels of shape (N,))."""
    N = logits.shape[0]
    return -np.mean(log_softmax(logits)[np.arange(N), y])


# A stress test: enormous logits break the naive formula but not the stable one.
big_logits = np.array([[1000.0, 0.0, -1000.0]])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(big_logits) / np.exp(big_logits).sum(axis=1, keepdims=True)
print("naive softmax  :", naive)
print("stable softmax :", softmax(big_logits))
print("stable CE for true class 0:", cross_entropy(big_logits, np.array([0])))
print("stable CE for true class 2:", cross_entropy(big_logits, np.array([2])))

The naive computation returns `nan` (infinity divided by infinity), while the stable version gives the
correct probabilities $(1, 0, 0)$ up to float precision and a finite loss of exactly $2000$ when the true
class is the one with logit $-1000$. Note how the loss keeps growing linearly as the model becomes more
confidently wrong, even though the probability has long underflowed to zero: this is why we always work
with log-probabilities.

### 4.3 Why not optimize accuracy directly?

Accuracy is what we ultimately care about, so why train with the cross-entropy? Because accuracy is a
**piecewise-constant** function of the parameters. Moving a weight slightly does not change any prediction
until a decision boundary crosses a data point, at which moment the accuracy jumps. Its derivative is
therefore zero almost everywhere and undefined at the jumps — gradient descent would have no signal to
follow. The cross-entropy is a smooth surrogate: it decreases continuously as the correct class's
probability grows, and it keeps rewarding improvements even for examples that are already classified
correctly but with low confidence. The next cell plots both quantities for a one-parameter classifier.

In [ ]:
rng = np.random.default_rng(SEED)
x_1d = rng.normal(size=60)                                  # 60 scalar inputs
y_1d = (x_1d + 0.6 * rng.normal(size=60) > 0).astype(int)   # noisy labels: class 1 when x is positive

w_grid = np.linspace(-2, 4, 300)
acc_curve, ce_curve = [], []
for w in w_grid:
    logits = np.stack([np.zeros_like(x_1d), w * x_1d], axis=1)   # two-class logits: class 1 score = w * x
    acc_curve.append(np.mean(logits.argmax(axis=1) == y_1d))
    ce_curve.append(cross_entropy(logits, y_1d))

fig, ax1 = plt.subplots()
ax1.plot(w_grid, ce_curve, color="C0", label="cross-entropy")
ax1.set_xlabel("weight $w$"); ax1.set_ylabel("cross-entropy", color="C0")
ax2 = ax1.twinx()
ax2.plot(w_grid, acc_curve, color="C3", label="accuracy")
ax2.set_ylabel("accuracy", color="C3"); ax2.grid(False)
ax1.set_title("a smooth loss versus a piecewise-constant metric")
fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88))
plt.tight_layout(); plt.show()

The accuracy (red) is a staircase: flat almost everywhere, with jumps wherever a data point changes side.
The cross-entropy (blue) is smooth and convex in $w$, with a well-defined minimum near the value that also
maximizes accuracy. Minimizing the cross-entropy is a proxy, but a well-behaved one: it is the loss we use
for every classifier in this course. We also keep track of the accuracy during training, purely as a
metric to report.

> **Common pitfall.** Do not compute the softmax and then take its logarithm in two separate steps
> (`np.log(softmax(Z))`). Where the probability underflows to zero the logarithm returns $-\infty$ and the
> training loss becomes `inf` or `nan`. Always use `log_softmax`, and — as Section 5 shows — never even form
> the probabilities in the backward pass.

## 5. Backpropagation, derived and checked

### 5.1 The computational graph and the chain rule

Training means finding parameters that make the loss small, and gradient descent needs the partial
derivative of the loss with respect to *every* parameter — for the MNIST network, 235,146 of them.
**Backpropagation** (Rumelhart, Hinton and Williams, 1986) computes all of them in one sweep at a cost
comparable to a single forward pass (about twice, as Section 5.5 shows). It is nothing more than the chain
rule, organised cleverly.

Think of the forward pass as a **computational graph**: a chain of operations, each taking the output of
the previous one,

$$
X \;\xrightarrow{\;W_1, \mathbf{b}_1\;}\; Z_1 \;\xrightarrow{\;\phi\;}\; A_1
\;\xrightarrow{\;W_2, \mathbf{b}_2\;}\; Z_2 \;\xrightarrow{\;\phi\;}\; A_2
\;\xrightarrow{\;W_3, \mathbf{b}_3\;}\; Z_3 \;\xrightarrow{\;\text{softmax-CE}\;}\; L .
$$

The loss $L$ is a scalar, and for each intermediate array $Q$ in the graph we define the **upstream
gradient** $G_Q = \partial L / \partial Q$, an array of *the same shape as $Q$* whose entry $(i, j)$ says how
much $L$ changes per unit change of $Q_{ij}$. The chain rule says that if $Q$ feeds into $R$ and we
already know $G_R$, then

$$
\frac{\partial L}{\partial Q_{ij}} = \sum_{k, l} \frac{\partial L}{\partial R_{kl}}\, \frac{\partial R_{kl}}{\partial Q_{ij}} .
$$

Backpropagation walks the graph *backwards*: starting from $G_L = 1$, each operation receives the gradient
with respect to its output and produces the gradient with respect to its inputs (and its parameters). We
therefore need one **backward rule** per type of operation. There are only three in our network — the
linear layer, the element-wise activation and the softmax cross-entropy — and we now derive each with
explicit shapes.

### 5.2 Backward through a linear layer

Consider $Z = X W + \mathbf{1}\mathbf{b}^\top$ with $X \in \mathbb{R}^{N \times d_{\text{in}}}$, $W \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$,
$\mathbf{b} \in \mathbb{R}^{d_{\text{out}}}$, $Z \in \mathbb{R}^{N \times d_{\text{out}}}$, and suppose we are given
$G_Z = \partial L / \partial Z \in \mathbb{R}^{N \times d_{\text{out}}}$. Element-wise, $Z_{nj} = \sum_i X_{ni} W_{ij} + b_j$.

*Gradient with respect to the weights.* $W_{ij}$ appears in $Z_{nj}$ for every example $n$ (and only in
column $j$), with $\partial Z_{nj} / \partial W_{ij} = X_{ni}$. Therefore

$$
\frac{\partial L}{\partial W_{ij}} = \sum_{n=1}^{N} \frac{\partial L}{\partial Z_{nj}}\, X_{ni} = \sum_n X_{ni}\, (G_Z)_{nj}
\quad\Longrightarrow\quad
\boxed{\;G_W = X^\top G_Z\;} \qquad (d_{\text{in}} \times N)(N \times d_{\text{out}}) = d_{\text{in}} \times d_{\text{out}} .
$$

The shape check is reassuring: $G_W$ has the shape of $W$, as it must. The sum over $n$ is the statement
that the batch contributes additively to the gradient — every example votes on how $W$ should change.

*Gradient with respect to the bias.* $b_j$ appears in $Z_{nj}$ for every $n$ with derivative 1, so

$$
\frac{\partial L}{\partial b_j} = \sum_n (G_Z)_{nj}
\quad\Longrightarrow\quad
\boxed{\;G_{\mathbf{b}} = G_Z^\top \mathbf{1} = \textstyle\sum_{\text{rows}} G_Z\;} \qquad (d_{\text{out}},) .
$$

*Gradient with respect to the input.* This is what gets passed further back. $X_{ni}$ appears in
$Z_{nj}$ for every $j$ (same row $n$) with derivative $W_{ij}$, so

$$
\frac{\partial L}{\partial X_{ni}} = \sum_j (G_Z)_{nj}\, W_{ij}
\quad\Longrightarrow\quad
\boxed{\;G_X = G_Z W^\top\;} \qquad (N \times d_{\text{out}})(d_{\text{out}} \times d_{\text{in}}) = N \times d_{\text{in}} .
$$

> **Key idea.** For a linear layer, the backward pass is two matrix products and a row-sum: $G_W = X^\top G_Z$,
> $G_{\mathbf{b}} = \sum_n (G_Z)_n$, $G_X = G_Z W^\top$. The forward pass must remember its input $X$; the backward
> pass needs nothing else. If you ever forget the formulas, the shapes almost force them on you: there is only
> one way to combine $X^\top$ and $G_Z$ into something of shape $d_{\text{in}} \times d_{\text{out}}$.

### 5.3 Backward through an element-wise activation

For $A = \phi(Z)$ applied element-wise, $A_{nj}$ depends only on $Z_{nj}$, so the chain rule has a single term:

$$
\frac{\partial L}{\partial Z_{nj}} = \frac{\partial L}{\partial A_{nj}}\, \phi'(Z_{nj})
\quad\Longrightarrow\quad
\boxed{\;G_Z = G_A \odot \phi'(Z)\;},
$$

where $\odot$ is the element-wise (Hadamard) product. For ReLU, $\phi'(Z) = \mathbb{1}[Z > 0]$: the backward
pass simply zeroes the upstream gradient wherever the unit was inactive and passes it through unchanged
elsewhere. For sigmoid, $\phi'(Z) = A \odot (1 - A)$ is computed from the stored output.

### 5.4 Backward through softmax cross-entropy: the $\mathbf{p} - \mathbf{y}$ result

The last piece is the gradient of the loss with respect to the logits. We compute it for one example
(row $n$ of $Z_3$) and then average. Write $\mathbf{z} \in \mathbb{R}^k$ for the logits, $y$ for the true class and
$\ell = -\log p_y = \log \sum_i e^{z_i} - z_y$. Differentiating with respect to $z_j$:

$$
\frac{\partial \ell}{\partial z_j} = \frac{e^{z_j}}{\sum_i e^{z_i}} - \mathbb{1}[j = y] = p_j - \mathbb{1}[j = y] .
$$

With the one-hot vector $\mathbf{y}$ (a 1 at position $y$, zeros elsewhere), this reads
$\partial \ell / \partial \mathbf{z} = \mathbf{p} - \mathbf{y}$: the gradient is *the predicted distribution minus the
target distribution*. It is remarkably clean. If the model is certain and correct ($\mathbf{p} = \mathbf{y}$)
the gradient vanishes; if it puts probability on a wrong class, a gradient step pushes that class's logit
down in proportion to $p_j$ and the true class's logit up in proportion to $1 - p_y$. Since the batch loss
averages $N$ such terms,

$$
\boxed{\;G_{Z_3} = \frac{1}{N}\,(P - Y)\;} \qquad (N \times k),
$$

where $P$ holds the softmax probabilities row by row and $Y$ the one-hot targets. The derivation also
explains why we never separate the softmax from the cross-entropy in code: the two Jacobians (that of
softmax is a full $k \times k$ matrix per example, $\partial p_i / \partial z_j = p_i(\delta_{ij} - p_j)$) cancel
to this simple expression. Exercise 6 asks you to do it the long way and confirm.

### 5.5 The complete backward pass

Putting the three rules together, the backward pass of the two-hidden-layer network is:

$$
\begin{aligned}
G_{Z_3} &= (P - Y)/N && (N \times k) \\
G_{W_3} = A_2^\top G_{Z_3}, \quad G_{\mathbf{b}_3} &= \textstyle\sum_n (G_{Z_3})_n, \quad G_{A_2} = G_{Z_3} W_3^\top && (d_2 \times k),\ (k,),\ (N \times d_2) \\
G_{Z_2} &= G_{A_2} \odot \mathbb{1}[Z_2 > 0] && (N \times d_2) \\
G_{W_2} = A_1^\top G_{Z_2}, \quad G_{\mathbf{b}_2} &= \textstyle\sum_n (G_{Z_2})_n, \quad G_{A_1} = G_{Z_2} W_2^\top && (d_1 \times d_2),\ (d_2,),\ (N \times d_1) \\
G_{Z_1} &= G_{A_1} \odot \mathbb{1}[Z_1 > 0] && (N \times d_1) \\
G_{W_1} = X^\top G_{Z_1}, \quad G_{\mathbf{b}_1} &= \textstyle\sum_n (G_{Z_1})_n . && (d_0 \times d_1),\ (d_1,)
\end{aligned}
$$

Notice the pattern: the backward pass visits the layers in reverse order, and the cost is dominated by two
matrix products per layer, each the same size as the forward product. Backpropagation therefore costs
about twice a forward pass, regardless of the number of parameters. Compare this with the alternative of
perturbing each parameter separately (finite differences), which would require one full forward pass *per
parameter* — 235,146 forward passes for one gradient of the MNIST network.

In [ ]:
def mlp_backward(params, cache, logits, y):
    """Backward pass matching mlp_forward. Returns dL/dparam for every parameter (same shapes)."""
    N = logits.shape[0]
    P = softmax(logits)
    G_Z3 = P.copy()
    G_Z3[np.arange(N), y] -= 1.0          # P - Y, using integer labels instead of a one-hot matrix
    G_Z3 /= N                             # (N, k)

    grads = {}
    grads["W3"] = cache["A2"].T @ G_Z3    # (d2, k)
    grads["b3"] = G_Z3.sum(axis=0)        # (k,)
    G_A2 = G_Z3 @ params["W3"].T          # (N, d2)
    G_Z2 = G_A2 * (cache["Z2"] > 0)       # (N, d2)   ReLU backward

    grads["W2"] = cache["A1"].T @ G_Z2    # (d1, d2)
    grads["b2"] = G_Z2.sum(axis=0)        # (d2,)
    G_A1 = G_Z2 @ params["W2"].T          # (N, d1)
    G_Z1 = G_A1 * (cache["Z1"] > 0)       # (N, d1)

    grads["W1"] = cache["X"].T @ G_Z1     # (d0, d1)
    grads["b1"] = G_Z1.sum(axis=0)        # (d1,)
    return grads


rng = np.random.default_rng(SEED)
params = init_mlp([4, 8, 6, 3], rng)
X_demo = rng.normal(size=(5, 4))
y_demo = rng.integers(0, 3, size=5)
logits, cache = mlp_forward(params, X_demo)
grads = mlp_backward(params, cache, logits, y_demo)

print(f"loss = {cross_entropy(logits, y_demo):.4f}")
for name in params:
    assert grads[name].shape == params[name].shape, name
    print(f"grad {name}: shape {str(grads[name].shape):8s} matches parameter shape")

### 5.6 Gradient checking

Shapes matching is necessary but not sufficient — a sign error or a forgotten ReLU mask produces
gradients of exactly the right shape and completely wrong values. The standard defence, introduced in
Notebook 01 · Mathematical Foundations, is **gradient checking**: compare the analytic gradient with the
central finite difference

$$
\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta + \varepsilon \mathbf{e}_i) - L(\theta - \varepsilon \mathbf{e}_i)}{2\varepsilon},
$$

for a random sample of parameter entries $i$. The central difference has an error of order $\varepsilon^2$, so
with $\varepsilon = 10^{-6}$ in float64 we expect agreement to about $10^{-8}$ or better. We report the
**relative error** $|g_{\text{num}} - g_{\text{ana}}| / (|g_{\text{num}}| + |g_{\text{ana}}|)$, which is insensitive to
the overall scale of the gradient; values below $10^{-6}$ indicate a correct implementation, values around
$10^{-2}$ or worse indicate a bug. Two practical remarks: always check in float64 (float32 has only about
seven significant digits, and the difference of two nearly equal losses loses most of them), and be
aware that a ReLU whose pre-activation lies within $\varepsilon$ of zero will produce a spurious error.

In [ ]:
def grad_check(loss_fn, params, grads, rng, n_checks=10, eps=1e-6):
    """Compare analytic grads with central differences on random entries of each parameter.

    loss_fn: function of no arguments that evaluates the loss with the *current* params
             (they are perturbed in place, then restored).
    Returns the largest relative error found.
    """
    worst = 0.0
    for name, p in params.items():
        errors = []
        for _ in range(n_checks):
            idx = tuple(rng.integers(0, s) for s in p.shape)
            original = p[idx]
            p[idx] = original + eps; loss_plus = loss_fn()
            p[idx] = original - eps; loss_minus = loss_fn()
            p[idx] = original                                  # restore
            numeric = (loss_plus - loss_minus) / (2 * eps)
            analytic = grads[name][idx]
            errors.append(abs(numeric - analytic) / max(abs(numeric) + abs(analytic), 1e-12))
        print(f"{name}: max relative error over {n_checks} entries = {max(errors):.2e}")
        worst = max(worst, max(errors))
    return worst


loss_fn = lambda: cross_entropy(mlp_forward(params, X_demo)[0], y_demo)
worst = grad_check(loss_fn, params, grads, rng)
print(f"\nworst relative error: {worst:.2e}  ->  {'PASS' if worst < 1e-6 else 'FAIL'}")

Every parameter agrees with the finite-difference estimate to a relative error below $10^{-7}$: the derivation and its
implementation are correct. To see that the check has teeth, let us introduce a classic bug — forgetting
the ReLU mask in the backward pass, i.e. treating the activation as if it were the identity — and run the
check again.

In [ ]:
def mlp_backward_buggy(params, cache, logits, y):
    """Same as mlp_backward but *without* the ReLU mask on the second hidden layer (a deliberate bug)."""
    N = logits.shape[0]
    P = softmax(logits)
    G_Z3 = P.copy(); G_Z3[np.arange(N), y] -= 1.0; G_Z3 /= N
    grads = {"W3": cache["A2"].T @ G_Z3, "b3": G_Z3.sum(axis=0)}
    G_Z2 = G_Z3 @ params["W3"].T                       # BUG: missing "* (cache['Z2'] > 0)"
    grads["W2"] = cache["A1"].T @ G_Z2; grads["b2"] = G_Z2.sum(axis=0)
    G_Z1 = (G_Z2 @ params["W2"].T) * (cache["Z1"] > 0)
    grads["W1"] = cache["X"].T @ G_Z1; grads["b1"] = G_Z1.sum(axis=0)
    return grads


grads_buggy = mlp_backward_buggy(params, cache, logits, y_demo)
worst = grad_check(loss_fn, params, grads_buggy, rng)
print(f"\nworst relative error: {worst:.2e}  ->  {'PASS' if worst < 1e-6 else 'FAIL'}")

The gradients of $W_3$ and $\mathbf{b}_3$ are still correct — they sit *after* the bug in the backward
order — but everything upstream of the missing mask is wrong by a relative error of order one. The
gradient checker localises the bug for free: the first parameter (in backward order) whose gradient
fails is next to the faulty rule. Get into the habit of gradient-checking every new layer you write;
Exercise 1 and Exercise 6 ask you to do exactly that.

> **Common pitfall.** Gradient checking must be done on a *tiny* network with float64 parameters and a
> handful of examples. On the real network each finite-difference sample costs two full forward passes,
> and in float32 the finite differences are dominated by rounding noise, so a correct implementation can
> look wrong.

## 6. A modular implementation

### 6.1 Layers as objects with `forward` and `backward`

The functions of Section 5 work, but they hard-code one architecture. Every deep learning framework
instead organises code around **layers**: objects that know how to compute their output from their input
(`forward`), how to turn a gradient with respect to their output into a gradient with respect to their
input (`backward`), and which parameters they own. A network is then just a list of layers, the forward
pass runs the list from left to right, and the backward pass runs it from right to left. This is exactly
the structure of `torch.nn`, and building it ourselves makes that library's design obvious later.

Our small library has the following pieces:

- `Layer`: a base class with dictionaries `params` and `grads`; layers without parameters leave them empty.
- `Linear`: owns `W` and `b`, remembers its input in `forward`, and implements the three formulas of Section 5.2.
- `ReLU`, `Sigmoid`: parameter-free activations that remember what they need for $\phi'$.
- `SoftmaxCrossEntropy` and `MSE`: loss objects whose `forward` returns the scalar loss and whose `backward` returns the gradient with respect to the network output, i.e. the first upstream gradient.
- `Sequential`: a container that chains `forward` and `backward` over its layers.
- `SGD`: an optimizer that subtracts `lr * grad` from every parameter in place.

The `Linear` layer takes a `dtype` argument. We use float32 by default — it halves the memory and roughly
doubles the speed of the MNIST experiments — and switch to float64 only when gradient-checking.

In [ ]:
class Layer:
    """Base class. Subclasses implement forward(x) -> out and backward(grad_out) -> grad_in."""

    def __init__(self):
        self.params, self.grads = {}, {}

    def forward(self, x):
        raise NotImplementedError

    def backward(self, grad_out):
        raise NotImplementedError

    def __call__(self, x):
        return self.forward(x)


class Linear(Layer):
    """Affine map x @ W + b with W of shape (fan_in, fan_out)."""

    def __init__(self, fan_in, fan_out, rng, init="he", dtype=np.float32):
        super().__init__()
        std = {"he": np.sqrt(2.0 / fan_in), "xavier": np.sqrt(2.0 / (fan_in + fan_out)), "small": 0.01, "zero": 0.0}[init]
        W = (rng.normal(size=(fan_in, fan_out)) * std).astype(dtype)
        self.params = {"W": W, "b": np.zeros(fan_out, dtype=dtype)}
        self.grads = {"W": np.zeros_like(W), "b": np.zeros(fan_out, dtype=dtype)}

    def forward(self, x):
        self.x = x                                             # needed for dW = x^T @ grad_out
        return x @ self.params["W"] + self.params["b"]

    def backward(self, grad_out):
        self.grads["W"] = self.x.T @ grad_out                  # (fan_in, fan_out)
        self.grads["b"] = grad_out.sum(axis=0)                 # (fan_out,)
        return grad_out @ self.params["W"].T                   # (N, fan_in)


class ReLU(Layer):
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, grad_out):
        return grad_out * self.mask


class Sigmoid(Layer):
    def forward(self, x):
        self.out = sigmoid(x)
        return self.out

    def backward(self, grad_out):
        return grad_out * self.out * (1.0 - self.out)


class SoftmaxCrossEntropy:
    """Combined softmax + cross-entropy loss for integer labels y of shape (N,)."""

    def forward(self, logits, y):
        self.y, self.N = y, logits.shape[0]
        log_p = log_softmax(logits)
        self.P = np.exp(log_p)
        return float(-np.mean(log_p[np.arange(self.N), y]))

    def backward(self):
        G = self.P.copy()
        G[np.arange(self.N), self.y] -= 1.0
        return G / self.N                                      # (P - Y) / N


class MSE:
    """Mean squared error with the 1/2 convention; targets have the same shape as the predictions."""

    def forward(self, y_hat, y):
        self.diff = y_hat - y
        return float(0.5 * np.mean(self.diff ** 2))

    def backward(self):
        return self.diff / self.diff.size


class Sequential:
    """A chain of layers. forward runs left to right, backward right to left."""

    def __init__(self, *layers):
        self.layers = list(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def backward(self, grad_out):
        for layer in reversed(self.layers):
            grad_out = layer.backward(grad_out)
        return grad_out

    def parameters(self):
        """Yield (layer, name, array) for every parameter, in forward order."""
        for layer in self.layers:
            for name, p in layer.params.items():
                yield layer, name, p

    def n_params(self):
        return sum(p.size for _, _, p in self.parameters())

    __call__ = forward


class SGD:
    """Plain mini-batch gradient descent: p <- p - lr * grad, in place."""

    def __init__(self, model, lr):
        self.model, self.lr = model, lr

    def step(self):
        for layer, name, p in self.model.parameters():
            p -= self.lr * layer.grads[name]


rng = np.random.default_rng(SEED)
demo = Sequential(Linear(4, 8, rng), ReLU(), Linear(8, 6, rng), ReLU(), Linear(6, 3, rng))
print(f"demo model: {len(demo.layers)} layers, {demo.n_params()} parameters")
print("output shape for a batch of 5:", demo(np.random.default_rng(0).normal(size=(5, 4)).astype(np.float32)).shape)

A subtle but important detail lives in `SGD.step`: the update `p -= self.lr * grad` modifies the parameter
array **in place**. Because `layer.params["W"]` and the `p` yielded by `parameters()` are the same NumPy
object, the layer sees the updated weights. Had we written `p = p - lr * grad`, NumPy would have created a
new array and bound the local name `p` to it, leaving the layer's weights untouched — the network would
never learn, and no error would tell us why. PyTorch has the same distinction (`p.sub_(...)` versus
`p = p - ...`), and we come back to it in Notebook 03 · Tensors and Autograd.

Before trusting the library, we confirm that it reproduces the hand-written functions of Section 5 on
the same parameters: same loss, same gradients.

In [ ]:
# Copy the Section 5 parameters into the modular model (float64 for an exact comparison).
rng = np.random.default_rng(SEED)
model64 = Sequential(Linear(4, 8, rng, dtype=np.float64), ReLU(), Linear(8, 6, rng, dtype=np.float64),
                     ReLU(), Linear(6, 3, rng, dtype=np.float64))
for layer, l in zip([model64.layers[0], model64.layers[2], model64.layers[4]], [1, 2, 3]):
    layer.params["W"][...] = params[f"W{l}"]
    layer.params["b"][...] = params[f"b{l}"]

loss_obj = SoftmaxCrossEntropy()
loss_modular = loss_obj.forward(model64(X_demo), y_demo)
model64.backward(loss_obj.backward())
loss_functional = cross_entropy(mlp_forward(params, X_demo)[0], y_demo)

print(f"loss: modular = {loss_modular:.8f}   functional = {loss_functional:.8f}")
for layer, l in zip([model64.layers[0], model64.layers[2], model64.layers[4]], [1, 2, 3]):
    same_W = np.allclose(layer.grads["W"], grads[f"W{l}"])
    same_b = np.allclose(layer.grads["b"], grads[f"b{l}"])
    print(f"layer {l}: gradients identical to Section 5 -> W: {same_W}, b: {same_b}")

### 6.2 The training loop

With layers, a loss and an optimizer in hand, training is a short loop. One pass over the training data
is an **epoch**; within an epoch the data is shuffled and cut into **mini-batches**, and each mini-batch
triggers one forward pass, one backward pass and one parameter update. We follow the pattern that every
later notebook uses — a `train_one_epoch` function, an `evaluate` function and a `fit` driver that logs
one line per epoch and returns the history for plotting. Mini-batching is not just a memory-saving
device: the noisy gradient of a small batch is much cheaper than the exact gradient of the full dataset,
and many noisy steps beat few exact ones (Notebook 04 · Training Deep Networks: Optimization,
Initialization, Normalization, Regularization quantifies the trade-off).

In [ ]:
def iterate_minibatches(X, y, batch_size, rng, shuffle=True):
    """Yield (X_batch, y_batch) pairs covering the data once, in random order if shuffle=True."""
    order = rng.permutation(len(X)) if shuffle else np.arange(len(X))
    for start in range(0, len(X), batch_size):
        idx = order[start:start + batch_size]
        yield X[idx], y[idx]


def train_one_epoch(model, loss_fn, optimizer, X, y, batch_size, rng, shuffle=True):
    """One pass over the data. Returns the average training loss."""
    total, count = 0.0, 0
    for xb, yb in iterate_minibatches(X, y, batch_size, rng, shuffle):
        loss = loss_fn.forward(model(xb), yb)      # forward
        model.backward(loss_fn.backward())          # backward: fills layer.grads
        optimizer.step()                            # update
        total += loss * len(xb); count += len(xb)
    return total / count


def evaluate(model, loss_fn, X, y):
    """Loss and accuracy on a dataset (no parameter updates)."""
    logits = model(X)
    loss = loss_fn.forward(logits, y)
    acc = float(np.mean(logits.argmax(axis=1) == y))
    return loss, acc


def fit(model, loss_fn, optimizer, X_train, y_train, X_val, y_val, epochs, batch_size, rng,
        shuffle=True, log_every=None):
    """Train for `epochs` epochs, logging one compact line per epoch. Returns a history dict."""
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    log_every = log_every or max(1, epochs // 10)
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, loss_fn, optimizer, X_train, y_train, batch_size, rng, shuffle)
        val_loss, val_acc = evaluate(model, loss_fn, X_val, y_val)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
        if epoch % log_every == 0 or epoch == epochs:
            print(f"epoch {epoch:4d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | "
                  f"val acc {val_acc:.4f} | {time.time() - t0:5.1f}s")
    return history


def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(history["train_loss"], label="train"); axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_title(f"loss {title}"); axes[0].legend()
    axes[1].plot(history["val_acc"], color="C2"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
    axes[1].set_title(f"validation accuracy {title}")
    plt.tight_layout(); plt.show()

### 6.3 Two moons

Our first real dataset is scikit-learn's `make_moons`: two interleaving half-circles with Gaussian noise, a
problem that is easy to look at and impossible for a linear classifier. We train a 2-32-32-2 network with
ReLU activations and the softmax cross-entropy (two classes, two logits), and plot the decision boundary.
The learning rate of 0.1 and batch size of 32 are reasonable defaults for a small problem like this one.

In [ ]:
X_moons, y_moons = make_moons(n_samples=600, noise=0.2, random_state=SEED)
X_moons = X_moons.astype(np.float32)
X_moons_tr, y_moons_tr = X_moons[:500], y_moons[:500]
X_moons_val, y_moons_val = X_moons[500:], y_moons[500:]

rng = np.random.default_rng(SEED)
moons_model = Sequential(Linear(2, 32, rng), ReLU(), Linear(32, 32, rng), ReLU(), Linear(32, 2, rng))
moons_loss = SoftmaxCrossEntropy()
EPOCHS_MOONS = 100 if FAST_RUN else 300
hist_moons = fit(moons_model, moons_loss, SGD(moons_model, lr=0.1), X_moons_tr, y_moons_tr,
                 X_moons_val, y_moons_val, epochs=EPOCHS_MOONS, batch_size=32, rng=rng)

plot_history(hist_moons, "(two moons)")
plot_decision_boundary(lambda G: moons_model(G).argmax(axis=1), X_moons, y_moons,
                       title=f"MLP on two moons (val acc {hist_moons['val_acc'][-1]:.2f})")
plt.tight_layout(); plt.show()

The loss falls quickly during the first epochs and then keeps creeping down as the network sharpens the
boundary; validation accuracy settles in the high nineties (the remaining errors are points that the
noise pushed across the other moon — no classifier can get them right). The decision boundary is a curve
made of straight segments: each hidden ReLU unit contributes a kink, and the two layers combine them into
a smooth-looking shape. This is the learned-features story of Section 1 at work, now with the features
discovered by gradient descent instead of by hand.

### 6.4 A three-class spiral

The spiral dataset (three arms twisted around the origin) is a harder, multi-class test: the classes are
not separable by anything simple, and a network must carve out three interleaved curved regions. We
generate it ourselves — every dataset in this course that is not a standard download is created in the
notebook — and train the same architecture with three output logits. We also use this dataset to compare
ReLU with sigmoid hidden units, which illustrates the saturation discussion of Section 2.

In [ ]:
def make_spiral(n_per_class=200, n_classes=3, noise=0.2, seed=SEED):
    """Three (or more) interleaved spiral arms, one per class. Returns X (N, 2) float32 and y (N,) int."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for c in range(n_classes):
        radius = np.linspace(0.05, 1.0, n_per_class)
        angle = np.linspace(c * 4.0, (c + 1) * 4.0, n_per_class)      # each arm sweeps 4 radians
        angle = angle + noise * rng.normal(size=n_per_class)
        X.append(np.stack([radius * np.sin(angle), radius * np.cos(angle)], axis=1))
        y.append(np.full(n_per_class, c))
    X, y = np.concatenate(X).astype(np.float32), np.concatenate(y)
    order = rng.permutation(len(X))
    return X[order], y[order]


X_spiral, y_spiral = make_spiral()
X_spiral_tr, y_spiral_tr = X_spiral[:480], y_spiral[:480]
X_spiral_val, y_spiral_val = X_spiral[480:], y_spiral[480:]
EPOCHS_SPIRAL = 150 if FAST_RUN else 600

spiral_results = {}
for act_name, Act in [("ReLU", ReLU), ("Sigmoid", Sigmoid)]:
    rng = np.random.default_rng(SEED)                       # re-seed: both runs start from the same weights
    model = Sequential(Linear(2, 64, rng), Act(), Linear(64, 64, rng), Act(), Linear(64, 3, rng))
    print(f"--- hidden activation: {act_name}")
    hist = fit(model, SoftmaxCrossEntropy(), SGD(model, lr=0.1), X_spiral_tr, y_spiral_tr,
               X_spiral_val, y_spiral_val, epochs=EPOCHS_SPIRAL, batch_size=32, rng=rng, log_every=EPOCHS_SPIRAL // 3)
    spiral_results[act_name] = (model, hist)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for name, (model, hist) in spiral_results.items():
    axes[0].plot(hist["train_loss"], label=f"{name} train"); axes[0].plot(hist["val_loss"], "--", label=f"{name} val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy"); axes[0].set_title("spiral: loss curves"); axes[0].legend()
for ax, (name, (model, hist)) in zip(axes[1:], spiral_results.items()):
    plot_decision_boundary(lambda G, m=model: m(G).argmax(axis=1), X_spiral, y_spiral, ax=ax,
                           title=f"{name} hidden units (val acc {hist['val_acc'][-1]:.2f})", padding=0.2)
plt.tight_layout(); plt.show()

Both networks start from identical weights (we re-seed before each run — a habit worth keeping whenever
two methods are compared), yet the ReLU network drives the loss down far faster and reaches a much
cleaner boundary. The sigmoid network is not broken, only slow: its hidden units saturate, the gradient
is multiplied by at most $0.25$ on its way through each sigmoid layer, and with plain SGD it needs many
more epochs to catch up. With `FAST_RUN = True` the difference is even more visible because the budget is smaller. The
ReLU boundary shows the spiral arms as regions bounded by straight segments — again the signature of
piecewise-linear networks.

> **Try it.** Increase `noise` in `make_spiral` to 0.5 and retrain. The validation accuracy drops; look at
> the decision boundary and decide whether the network is under-fitting the arms or fitting the noise. Then
> reduce the hidden width to 8 and observe how the boundary simplifies.

## 7. MNIST in NumPy

### 7.1 Loading the data

MNIST (LeCun et al., 1998) is the classic benchmark of 70,000 handwritten digits, $28 \times 28$ grey-scale
pixels each, split into 60,000 training and 10,000 test images. It is small enough to train on a laptop
and rich enough that the choices we make — learning rate, batch size, initialization — visibly matter.
We use torchvision only as a downloader: the dataset object stores the images as a tensor of unsigned
bytes, which we convert to a NumPy array of shape $(N, 784)$ with pixel values in $[0, 1]$. The download
goes into `DATA_DIR` and is skipped when the files are already there. If the download is impossible
(no network), the loader falls back to scikit-learn's built-in `digits` dataset — 1,797 images of
$8 \times 8$ pixels — so that every cell below still runs; the code is written to work with either.

We then **standardize** the inputs by subtracting the training mean and dividing by the training standard
deviation. Standardized inputs keep the pre-activations of the first layer at a sensible scale (the He
initialization assumes inputs of roughly unit variance), and they make the same learning rate work
across datasets. The statistics are computed on the training set only and reused for validation and test
data, exactly as one would in a deployed system.

In [ ]:
def load_mnist_numpy(data_dir):
    """Return (X_train, y_train, X_test, y_test, image_shape, name) as NumPy arrays with X in [0, 1].

    Uses torchvision to download MNIST into data_dir; falls back to sklearn's 8x8 digits offline.
    """
    try:
        import torchvision
        train = torchvision.datasets.MNIST(root=str(data_dir), train=True, download=True)
        test = torchvision.datasets.MNIST(root=str(data_dir), train=False, download=True)
        # .data / .targets are uint8 / int64 CPU tensors; .cpu() is a no-op here, kept for the
        # course-wide habit of moving every tensor to the CPU before converting it to NumPy.
        X_tr = train.data.cpu().numpy().reshape(-1, 28 * 28).astype(np.float32) / 255.0
        y_tr = train.targets.cpu().numpy().astype(np.int64)
        X_te = test.data.cpu().numpy().reshape(-1, 28 * 28).astype(np.float32) / 255.0
        y_te = test.targets.cpu().numpy().astype(np.int64)
        return X_tr, y_tr, X_te, y_te, (28, 28), "MNIST"
    except Exception as err:
        print(f"Could not load MNIST through torchvision ({err!r}).")
        print("Falling back to sklearn's 8x8 digits so that the notebook still runs offline.")
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
        digits = load_digits()
        X = (digits.data / 16.0).astype(np.float32)
        y = digits.target.astype(np.int64)
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
        return X_tr, y_tr, X_te, y_te, (8, 8), "sklearn digits"


X_train_full, y_train_full, X_test, y_test, IMG_SHAPE, DATASET_NAME = load_mnist_numpy(DATA_DIR)
PIXEL_MEAN, PIXEL_STD = X_train_full.mean(), X_train_full.std()
X_train_full = (X_train_full - PIXEL_MEAN) / PIXEL_STD
X_test = (X_test - PIXEL_MEAN) / PIXEL_STD

N_VAL = min(5_000, len(X_train_full) // 5)
X_val, y_val = X_train_full[-N_VAL:], y_train_full[-N_VAL:]
N_TRAIN = min(10_000 if FAST_RUN else 55_000, len(X_train_full) - N_VAL)
X_train, y_train = X_train_full[:N_TRAIN], y_train_full[:N_TRAIN]
D_IN = X_train.shape[1]

print(f"{DATASET_NAME}: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}, "
      f"pixel mean {PIXEL_MEAN:.4f}, std {PIXEL_STD:.4f}")
fig, axes = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, img, label in zip(axes.ravel(), X_train[:16], y_train[:16]):
    ax.imshow(img.reshape(IMG_SHAPE), cmap="gray"); ax.set_title(int(label), fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

### 7.2 Training a 784-256-128-10 network

The architecture is the one whose parameters we counted in Section 3: two hidden ReLU layers of 256 and
128 units, ten output logits, 235,146 parameters in total. We train with plain mini-batch SGD at a
learning rate of 0.1 with batches of 64 images. With the full budget (`FAST_RUN = False`: 55,000 training
images, 10 epochs) this reaches about 97–98% test accuracy in about a minute of NumPy time; under
`FAST_RUN` we use a 10,000-image subset and a single epoch, which already gives roughly 90%. Either way the
numbers are worth pausing over: a few dozen lines of NumPy, no feature engineering, and the machine reads
digits better than most rule-based systems ever did.

In [ ]:
rng = np.random.default_rng(SEED)
mnist_model = Sequential(Linear(D_IN, 256, rng), ReLU(), Linear(256, 128, rng), ReLU(), Linear(128, 10, rng))
mnist_loss = SoftmaxCrossEntropy()
EPOCHS_MNIST = 1 if FAST_RUN else 10
print(f"{mnist_model.n_params():,} parameters | {len(X_train):,} training images | {EPOCHS_MNIST} epoch(s)")

hist_mnist = fit(mnist_model, mnist_loss, SGD(mnist_model, lr=0.1), X_train, y_train, X_val, y_val,
                 epochs=EPOCHS_MNIST, batch_size=64, rng=rng, log_every=1)
test_loss, test_acc = evaluate(mnist_model, mnist_loss, X_test, y_test)
print(f"\ntest loss {test_loss:.4f} | test accuracy {test_acc:.4f}")
if EPOCHS_MNIST > 1:
    plot_history(hist_mnist, "(MNIST)")

Training loss and validation loss fall together during the first epochs; towards the end the training
loss keeps decreasing while the validation loss flattens — the first sign of **overfitting**, which
Notebook 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization treats
in depth. The test accuracy lands close to the validation accuracy, as it should when
the validation set was never used for parameter updates.

### 7.3 Where does the network go wrong?

Accuracy is a single number; a **confusion matrix** shows *which* classes are confused with which. Entry
$(i, j)$ counts test images of true class $i$ that were predicted as class $j$, so a perfect classifier
has a diagonal matrix. We also display the highest-loss mistakes, which are often genuinely ambiguous
digits.

In [ ]:
def confusion_matrix(y_true, y_pred, n_classes=10):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    np.add.at(cm, (y_true, y_pred), 1)
    return cm


pred_test = mnist_model(X_test).argmax(axis=1)
cm = confusion_matrix(y_test, pred_test)
per_class_acc = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        if i != j and cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="black")
ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.grid(False)
ax.set_xlabel("predicted digit"); ax.set_ylabel("true digit"); ax.set_title("confusion matrix (off-diagonal counts shown)")
plt.colorbar(im, ax=ax, label="count"); plt.tight_layout(); plt.show()
print("per-class accuracy:", " ".join(f"{d}:{a:.3f}" for d, a in enumerate(per_class_acc)))

wrong = np.where(pred_test != y_test)[0]
fig, axes = plt.subplots(2, 6, figsize=(11, 4.2))
for ax, i in zip(axes.ravel(), wrong[:12]):
    ax.imshow(X_test[i].reshape(IMG_SHAPE), cmap="gray")
    ax.set_title(f"true {y_test[i]} / pred {pred_test[i]}", fontsize=9); ax.axis("off")
plt.suptitle(f"{len(wrong)} of {len(y_test)} test images misclassified"); plt.tight_layout(); plt.show()

The confusions are the ones a human would expect: 4 and 9, 3 and 5, 7 and 2, 8 and 3 — pairs of digits that
share most of their strokes. The misclassified images include some that are simply badly written and a few
where the label itself is debatable. Error analysis of this kind is the most under-used tool in machine
learning: it tells you whether to collect more data, clean labels, or change the model, and we return to it
in Notebook 05 · The PyTorch Workflow and Best Practices.

### 7.4 Learning rate, batch size and initialization

Three hyperparameters shape the outcome of every training run, and it pays to develop intuition for them
on a problem where experiments are cheap. The helper below trains a fresh network from the same seed under a
given configuration (a subset of the data and a small number of epochs, so that the whole study takes
seconds) and reports the final training loss and validation accuracy.

- **Learning rate.** Too small and training is slow; too large and the updates overshoot — the loss may
  oscillate, explode to `nan`, or push units into the dead-ReLU regime described in Section 2. The usable
  range spans roughly one order of magnitude, and finding it is always the first tuning step.
- **Batch size.** With a fixed number of epochs, a smaller batch means more (noisier) updates and usually a
  better result per epoch, at a higher wall-clock cost per epoch because small matrix products use the
  hardware poorly. Larger batches are faster per epoch but need more epochs (or a larger learning rate).
- **Initialization.** Zero initialization makes every hidden unit identical: they receive identical
  gradients and stay identical forever, so the network is effectively one unit wide — the **symmetry
  problem**. Too-small random weights shrink the signal layer by layer and the network starts on a plateau;
  He and Xavier initializations keep the signal alive.

In [ ]:
def run_experiment(lr=0.1, batch_size=64, init="he", epochs=None, n_train=None, seed=SEED):
    """Train a fresh 256-128 network on a subset with the given settings; return summary numbers."""
    epochs = epochs or (1 if FAST_RUN else 3)
    n_train = n_train or min(3_000 if FAST_RUN else 20_000, len(X_train))
    rng = np.random.default_rng(seed)
    model = Sequential(Linear(D_IN, 256, rng, init=init), ReLU(), Linear(256, 128, rng, init=init),
                       ReLU(), Linear(128, 10, rng, init=init))
    loss_fn, opt = SoftmaxCrossEntropy(), SGD(model, lr=lr)
    t0 = time.time()
    for _ in range(epochs):
        train_loss = train_one_epoch(model, loss_fn, opt, X_train[:n_train], y_train[:n_train], batch_size, rng)
    val_loss, val_acc = evaluate(model, loss_fn, X_val, y_val)
    return train_loss, val_acc, time.time() - t0


print(f"{'setting':34s} {'train loss':>10s} {'val acc':>8s} {'seconds':>8s}")
for lr in [0.01, 0.1, 1.0]:
    tl, va, dt = run_experiment(lr=lr)
    print(f"{'learning rate ' + str(lr):34s} {tl:10.4f} {va:8.4f} {dt:8.1f}")
for bs in [8, 64, 512]:
    tl, va, dt = run_experiment(batch_size=bs)
    print(f"{'batch size ' + str(bs):34s} {tl:10.4f} {va:8.4f} {dt:8.1f}")
for init in ["zero", "small", "xavier", "he"]:
    tl, va, dt = run_experiment(init=init)
    print(f"{'init ' + init:34s} {tl:10.4f} {va:8.4f} {dt:8.1f}")

Read the table row by row. A learning rate of 0.01 learns, but slowly; 0.1 is the sweet spot for this
network; at 1.0 training is unstable — the loss either explodes (possibly to `nan`) or the network
collapses to predicting a single class, and the validation accuracy is close to chance. Batch size 8
gives the best accuracy per epoch but takes the longest, while batch size 512 makes so few updates that
it lags far behind at equal epochs. Zero initialization is stuck near 10% accuracy: only the output bias
can learn, so the network predicts the most frequent class. The "small" initialization (std 0.01) starts
on a plateau — the signal is attenuated by two orders of magnitude on the way to the logits and the early
gradients are tiny — and on the `FAST_RUN` budget it may not leave the plateau at all. Xavier and He
initializations both work well here; the difference between them grows with depth, as Notebook 04 shows.

> **Try it.** Run `run_experiment(lr=0.3)` and `run_experiment(lr=0.03)`. Where exactly does the usable
> range end for this network? Then double the batch size *and* the learning rate together — the classic
> linear scaling rule — and see whether the accuracy per epoch is preserved.

## 8. Universal approximation in practice

### 8.1 One hidden layer can fit anything — in principle

The **universal approximation theorem** (Cybenko, 1989, for sigmoid units; Hornik, 1991, for any bounded
non-constant activation; Leshno et al., 1993, for any non-polynomial activation, ReLU included) states that
a network with a single hidden layer can approximate any continuous function on a compact set to
arbitrary accuracy, provided the hidden layer is wide enough. For ReLU networks the intuition is concrete:
a one-hidden-layer ReLU network computes

$$
f(x) = \sum_{j=1}^{H} v_j \max(0, w_j x + b_j) + c,
$$

a continuous **piecewise-linear** function with at most $H$ kinks, located at $x = -b_j / w_j$. Any
continuous function on an interval can be approximated by such a polyline, and more kinks give a finer
approximation. The theorem, however, says nothing about how wide the layer must be, nor whether gradient
descent will find the right weights. The experiment below fits $\sin(x)$ and a step function with widths
2, 4, 16 and 128 and lets you judge both questions empirically.

Two practical details deserve a comment. First, the kinks. Our `Linear` layer initialises every bias to
zero, so at the start *all* $H$ kinks sit at $x = -b_j / w_j = 0$ and the network is a "V" no matter how
wide it is; gradient descent then has to drag every kink to a useful place, and for a wide layer the
biases move very slowly (their gradient is proportional to the small output weights $v_j$). We therefore
draw a kink position $u_j$ uniformly over the input interval for every hidden unit and set
$b_j = -w_j u_j$, which starts the network as a random polyline whose kinks already cover the interval — a
data-dependent initialization that matches the features to the range of the inputs, in the same spirit as
the input standardization of Section 7. Second, the learning rate. We train with mean squared error and plain SGD, and the learning rate is set to `0.2 / width`. The
reason is that the loss surface of the output layer becomes sharper as the hidden layer grows — with $H$
hidden features of similar scale, the curvature along the direction that moves all output weights
together grows roughly in proportion to $H$ — and gradient descent diverges when the learning rate exceeds
about twice the inverse of that curvature. Adaptive optimizers such as Adam (Notebook 04 · Training Deep
Networks: Optimization, Initialization, Normalization, Regularization) largely remove this bookkeeping;
with plain SGD we have to do it ourselves.

In [ ]:
def fit_regression(x, y, width, epochs, lr, seed=SEED):
    """Train a 1-hidden-layer ReLU network on (x, y) with MSE. Returns the model and its final loss."""
    rng = np.random.default_rng(seed)
    model = Sequential(Linear(1, width, rng), ReLU(), Linear(width, 1, rng))
    # Spread the kinks x = -b_j / w_j uniformly over the input interval instead of piling them up at 0.
    hidden = model.layers[0]
    kinks = rng.uniform(x.min(), x.max(), size=width).astype(hidden.params["b"].dtype)
    hidden.params["b"][:] = -hidden.params["W"][0] * kinks        # W has shape (1, width)
    loss_fn, opt = MSE(), SGD(model, lr=lr)
    for _ in range(epochs):
        for xb, yb in iterate_minibatches(x, y, 32, rng):
            loss_fn.forward(model(xb), yb)
            model.backward(loss_fn.backward())
            opt.step()
    return model, loss_fn.forward(model(x), y)


x_reg = np.linspace(-np.pi, np.pi, 256, dtype=np.float32).reshape(-1, 1)
targets = {"sin(x)": np.sin(x_reg), "step(x)": (x_reg > 0).astype(np.float32)}
widths = [2, 4, 16, 128]
EPOCHS_REG = 300 if FAST_RUN else 1500

fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
for row, (target_name, y_reg) in enumerate(targets.items()):
    for ax, width in zip(axes[row], widths):
        model, final_loss = fit_regression(x_reg, y_reg, width, EPOCHS_REG, lr=0.2 / width)
        ax.plot(x_reg, y_reg, "k--", label="target")
        ax.plot(x_reg, model(x_reg), label="1-hidden-layer ReLU MLP")
        ax.set_title(f"{target_name}, width {width}, MSE {final_loss:.4f}", fontsize=10)
        ax.set_xlabel("x")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

With two hidden units the network can only produce two kinks, so $\sin(x)$ becomes a crude tent and the
step becomes a ramp. Four units already capture the general shape; sixteen give a visibly good fit; and
with 128 units the polyline follows the sine closely — more kinks, finer approximation, exactly as the
theorem promises. The step function is harder for a fixed width: a jump needs two kinks very close together
with a steep segment in between, which requires large weights that gradient descent approaches only
slowly — the fit keeps a small ramp around zero even at width 128. The theorem is true, but capacity and
trainability are different things.

### 8.2 Depth versus width, and why "from scratch" stops here

If one hidden layer suffices, why do we build deep networks? Because depth is *exponentially* more
efficient for many functions. A one-hidden-layer ReLU network with $H$ units has at most $H$ kinks, while
a network with $L$ layers of $H$ units can compose them into on the order of $H^L$ linear regions
(Montúfar et al., 2014); there exist simple functions — a sawtooth with $2^L$ teeth is the standard
example — that a depth-$L$ network represents with $O(L)$ units but that any shallow network needs
exponentially many units to approximate (Telgarsky, 2016). Depth also matches the structure of real data:
edges compose into parts, parts into objects, characters into words — each layer can build on the
abstractions of the previous one, which is the learned-features picture of Section 1 iterated. Notebook
04 and Notebook 06 · Convolutional Neural Networks show the practical side: deep networks are harder to
train, and much of the last decade's
progress consisted of making depth trainable (initialization, normalization, residual connections).

This notebook also marks the end of the from-scratch approach. Look back at what adding a layer type
costs: a `forward`, a `backward` derived by hand, a gradient check, and the discipline to keep the cache
consistent. That was fine for `Linear` and `ReLU`. A convolution, a recurrent cell, an attention block or
a normalization layer each need their own carefully derived backward pass, and any change to the
architecture — a skip connection, a shared weight, a loss that combines several outputs — means
re-deriving gradients through a graph that is no longer a simple chain. NumPy also runs only on the CPU
and in the precision we ask for. The remedy is **automatic differentiation**: let the framework record
the computational graph as the forward pass runs and derive the backward pass mechanically, for any graph,
on any device. That is exactly what Notebook 03 · Tensors and Autograd builds — first as a tiny engine of
our own, then with `torch.autograd`. Everything you derived here is what those engines do internally;
you will recognise the $X^\top G$ and $G W^\top$ products in every framework's `Linear` backward.

> **Key idea.** Backpropagation is the chain rule applied to a computational graph in reverse order. Doing it
> by hand teaches what the framework does; letting the framework do it is what makes modern architectures
> possible.

## Exercises

The exercises extend the small library of Section 6. Each solution is complete and runnable; try to write
your own version before reading it, and use the gradient checker whenever you add a backward pass.

### Exercise 1 (warm-up): a `Tanh` layer

Implement a `Tanh` layer with `forward` and `backward` (recall $\tanh'(z) = 1 - \tanh^2(z)$). Then write a
helper `model_grad_check(model, loss_fn, X, y, rng)` that gradient-checks *every* parameter of a
`Sequential` model, and use it to verify a 4-8-3 network with a `Tanh` hidden layer in float64.

**Solution to Exercise 1**

The backward pass reuses the stored output, so no extra `tanh` evaluation is needed. The checker mirrors
`grad_check` from Section 5.6 but walks the model's `parameters()`; the analytic gradients are computed
once, before the parameters are perturbed, and read back from `layer.grads`.

In [ ]:
class Tanh(Layer):
    def forward(self, x):
        self.out = np.tanh(x)
        return self.out

    def backward(self, grad_out):
        return grad_out * (1.0 - self.out ** 2)


def model_grad_check(model, loss_fn, X, y, rng, n_checks=5, eps=1e-6):
    """Gradient-check every parameter of a Sequential model. Use float64 layers and inputs."""
    loss_fn.forward(model(X), y)
    model.backward(loss_fn.backward())                 # analytic gradients are now in layer.grads
    worst = 0.0
    for layer, name, p in model.parameters():
        for _ in range(n_checks):
            idx = tuple(rng.integers(0, s) for s in p.shape)
            original = p[idx]
            p[idx] = original + eps; loss_plus = loss_fn.forward(model(X), y)
            p[idx] = original - eps; loss_minus = loss_fn.forward(model(X), y)
            p[idx] = original
            numeric = (loss_plus - loss_minus) / (2 * eps)
            analytic = layer.grads[name][idx]
            worst = max(worst, abs(numeric - analytic) / max(abs(numeric) + abs(analytic), 1e-12))
    return worst


rng = np.random.default_rng(SEED)
tanh_model = Sequential(Linear(4, 8, rng, dtype=np.float64), Tanh(), Linear(8, 3, rng, dtype=np.float64))
X_chk = rng.normal(size=(6, 4)); y_chk = rng.integers(0, 3, size=6)
worst = model_grad_check(tanh_model, SoftmaxCrossEntropy(), X_chk, y_chk, rng)
print(f"Tanh network: worst relative error {worst:.2e} -> {'PASS' if worst < 1e-6 else 'FAIL'}")

# A quick functional test: tanh hidden units on the two moons.
rng = np.random.default_rng(SEED)
m = Sequential(Linear(2, 32, rng), Tanh(), Linear(32, 32, rng), Tanh(), Linear(32, 2, rng))
h = fit(m, SoftmaxCrossEntropy(), SGD(m, lr=0.1), X_moons_tr, y_moons_tr, X_moons_val, y_moons_val,
        epochs=50 if FAST_RUN else 150, batch_size=32, rng=rng, log_every=1000)

The relative error is around $10^{-9}$, so the layer is correct, and the tanh network trains on the moons
as well as the ReLU one did. The `Tanh` layer will be handy in Notebook 08 · Sequence Models: RNNs, LSTMs
and GRUs, where it is the traditional activation of recurrent cells.

### Exercise 2: SGD with momentum

Implement `MomentumSGD`, which keeps a velocity $\mathbf{v}$ per parameter and updates
$\mathbf{v} \leftarrow \mu \mathbf{v} - \eta\, \mathbf{g}$, $\theta \leftarrow \theta + \mathbf{v}$ with momentum
$\mu = 0.9$. Compare it with plain SGD on the spiral dataset (same seed, same number of epochs) by plotting
the two training-loss curves. Use a smaller learning rate for momentum — why?

**Solution to Exercise 2**

With momentum the effective step in a direction of consistent gradient grows to $\eta / (1 - \mu)$, ten
times the nominal learning rate for $\mu = 0.9$; to compare at equal effective step size we use
$\eta = 0.02$ for momentum against $\eta = 0.1$ for plain SGD (a factor of five rather than ten, which also
tests whether momentum helps beyond a mere rescaling). Notebook 04 · Training Deep Networks: Optimization,
Initialization, Normalization, Regularization derives the update properly and adds Nesterov's variant.

In [ ]:
class MomentumSGD:
    """SGD with (heavy-ball) momentum: v <- mu * v - lr * grad, p <- p + v."""

    def __init__(self, model, lr, momentum=0.9):
        self.model, self.lr, self.momentum = model, lr, momentum
        self.velocity = {(id(layer), name): np.zeros_like(p) for layer, name, p in model.parameters()}

    def step(self):
        for layer, name, p in self.model.parameters():
            v = self.velocity[(id(layer), name)]
            v *= self.momentum
            v -= self.lr * layer.grads[name]
            p += v


curves = {}
for opt_name in ["SGD lr=0.1", "momentum lr=0.02, mu=0.9"]:
    rng = np.random.default_rng(SEED)
    model = Sequential(Linear(2, 64, rng), ReLU(), Linear(64, 64, rng), ReLU(), Linear(64, 3, rng))
    opt = SGD(model, lr=0.1) if opt_name.startswith("SGD") else MomentumSGD(model, lr=0.02, momentum=0.9)
    hist = fit(model, SoftmaxCrossEntropy(), opt, X_spiral_tr, y_spiral_tr, X_spiral_val, y_spiral_val,
               epochs=EPOCHS_SPIRAL // 2, batch_size=32, rng=rng, log_every=1000)
    curves[opt_name] = hist
    print(f"{opt_name:28s} final train loss {hist['train_loss'][-1]:.4f}, val acc {hist['val_acc'][-1]:.3f}")

plt.figure()
for name, hist in curves.items():
    plt.plot(hist["train_loss"], label=name)
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("spiral: plain SGD vs momentum"); plt.legend()
plt.tight_layout(); plt.show()

Momentum reaches a lower loss in fewer epochs even though its nominal learning rate is five times smaller:
the velocity accumulates gradient components that point consistently in the same direction and averages
out the components that flip sign from mini-batch to mini-batch. This is the single most useful addition
to SGD, and `torch.optim.SGD(momentum=0.9)` is the standard way to enable it in PyTorch (the optimizer's own default is `momentum=0`).

### Exercise 3: L2 regularization in the backward pass

Add an L2 penalty $\frac{\lambda}{2}\|W\|_F^2$ for every weight matrix (biases excluded) by modifying the
backward pass of the linear layer: the gradient of the penalty is simply $\lambda W$. Train on a noisier
version of the two moons (`noise=0.35`) with $\lambda \in \{0, 0.005, 0.05\}$ and compare the decision
boundaries and validation accuracies.

**Solution to Exercise 3**

The cleanest way is a subclass that adds $\lambda W$ to `grads["W"]` after the ordinary backward pass. The
loss values printed during training do *not* include the penalty (we only changed the gradient), which is
the usual convention in frameworks too — PyTorch's `weight_decay` option behaves exactly like this.

In [ ]:
class LinearL2(Linear):
    """Linear layer whose backward pass adds the gradient of (weight_decay / 2) * ||W||^2."""

    def __init__(self, fan_in, fan_out, rng, weight_decay=0.0, **kwargs):
        super().__init__(fan_in, fan_out, rng, **kwargs)
        self.weight_decay = weight_decay

    def backward(self, grad_out):
        grad_in = super().backward(grad_out)
        self.grads["W"] = self.grads["W"] + self.weight_decay * self.params["W"]
        return grad_in


X_noisy, y_noisy = make_moons(n_samples=600, noise=0.35, random_state=SEED)
X_noisy = X_noisy.astype(np.float32)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, wd in zip(axes, [0.0, 0.005, 0.05]):
    rng = np.random.default_rng(SEED)
    model = Sequential(LinearL2(2, 64, rng, wd), ReLU(), LinearL2(64, 64, rng, wd), ReLU(), LinearL2(64, 2, rng, wd))
    hist = fit(model, SoftmaxCrossEntropy(), SGD(model, lr=0.1), X_noisy[:500], y_noisy[:500],
               X_noisy[500:], y_noisy[500:], epochs=EPOCHS_MOONS, batch_size=32, rng=rng, log_every=1000)
    plot_decision_boundary(lambda G, m=model: m(G).argmax(axis=1), X_noisy, y_noisy, ax=ax,
                           title=f"weight decay {wd}: val acc {hist['val_acc'][-1]:.2f}")
plt.tight_layout(); plt.show()

Without regularization the network bends its boundary around individual noisy points; a small penalty
smooths the boundary and typically improves the validation accuracy slightly; a large penalty shrinks the
weights so much that the boundary becomes nearly linear and the network under-fits. Regularization is a
dial between these two failure modes, and Notebook 04 shows how to set it.

### Exercise 4: does shuffling matter?

Sort a subset of the MNIST training set by label, so that all zeros come first, then all ones, and so on.
Train one network for one epoch on this sorted data *without* shuffling (`shuffle=False`), and another
with shuffling, from the same seed. Compare validation accuracies and explain the result.

**Solution to Exercise 4**

`iterate_minibatches` already has a `shuffle` flag, so the experiment only needs sorted data.

In [ ]:
n_sub = min(3_000 if FAST_RUN else 10_000, len(X_train))
order = np.argsort(y_train[:n_sub], kind="stable")
X_sorted, y_sorted = X_train[:n_sub][order], y_train[:n_sub][order]

for shuffle in [False, True]:
    rng = np.random.default_rng(SEED)
    model = Sequential(Linear(D_IN, 256, rng), ReLU(), Linear(256, 128, rng), ReLU(), Linear(128, 10, rng))
    hist = fit(model, SoftmaxCrossEntropy(), SGD(model, lr=0.1), X_sorted, y_sorted, X_val, y_val,
               epochs=1, batch_size=64, rng=rng, shuffle=shuffle, log_every=1000)
    preds = model(X_val).argmax(axis=1)
    print(f"shuffle={str(shuffle):5s}: val acc {hist['val_acc'][-1]:.4f} | "
          f"most predicted class: {np.bincount(preds, minlength=10).argmax()} "
          f"({np.bincount(preds, minlength=10).max() / len(preds):.0%} of predictions)")

Without shuffling the network sees hundreds of consecutive batches containing only one digit. Each phase
overwrites what the previous one learned — by the end of the epoch the network mostly predicts the last
class it saw — and the validation accuracy collapses. This is a small-scale instance of **catastrophic
forgetting**, and it is why every data loader shuffles by default: SGD's convergence guarantees assume that
each mini-batch is an unbiased sample of the whole training distribution.

### Exercise 5: a two-layer network for regression

Generate a regression dataset with two inputs $x_1, x_2 \sim \mathcal{U}(-2, 2)$ and target
$y = \sin(2x_1) + \tfrac{1}{2} x_2^2 + \epsilon$ with $\epsilon \sim \mathcal{N}(0, 0.1^2)$. Train a network
with one hidden layer of 64 ReLU units and a single linear output using the `MSE` loss, and report the
coefficient of determination $R^2 = 1 - \sum (y - \hat{y})^2 / \sum (y - \bar{y})^2$ on a held-out test set.

**Solution to Exercise 5**

Two details matter. The target is standardized (zero mean, unit variance) so that the MSE is on a sensible
scale, and the learning rate is small (0.01) for the reason discussed in Section 8: with 64 hidden features
the curvature of the output layer's loss is large. The training loop does not use `fit` because that helper
computes a classification accuracy; a five-line loop suffices.

In [ ]:
rng = np.random.default_rng(SEED)
X_r = rng.uniform(-2, 2, size=(1000, 2)).astype(np.float32)
y_r = np.sin(2 * X_r[:, :1]) + 0.5 * X_r[:, 1:] ** 2 + 0.1 * rng.normal(size=(1000, 1)).astype(np.float32)
y_mean, y_std = y_r[:800].mean(), y_r[:800].std()
y_r = ((y_r - y_mean) / y_std).astype(np.float32)
X_r_tr, y_r_tr, X_r_te, y_r_te = X_r[:800], y_r[:800], X_r[800:], y_r[800:]

reg_model = Sequential(Linear(2, 64, rng), ReLU(), Linear(64, 1, rng))
reg_loss, reg_opt = MSE(), SGD(reg_model, lr=0.01)
EPOCHS_R = 100 if FAST_RUN else 300
for epoch in range(1, EPOCHS_R + 1):
    for xb, yb in iterate_minibatches(X_r_tr, y_r_tr, 32, rng):
        reg_loss.forward(reg_model(xb), yb); reg_model.backward(reg_loss.backward()); reg_opt.step()
    if epoch % (EPOCHS_R // 4) == 0:
        print(f"epoch {epoch:4d} | train MSE {reg_loss.forward(reg_model(X_r_tr), y_r_tr):.4f} | "
              f"test MSE {reg_loss.forward(reg_model(X_r_te), y_r_te):.4f}")

y_pred = reg_model(X_r_te)
r2 = 1.0 - np.sum((y_r_te - y_pred) ** 2) / np.sum((y_r_te - y_r_te.mean()) ** 2)
print(f"test R^2 = {r2:.4f}")
plt.figure(figsize=(4.5, 4.5))
plt.scatter(y_r_te, y_pred, s=10); plt.plot([-3, 3], [-3, 3], "k--")
plt.xlabel("true (standardized)"); plt.ylabel("predicted"); plt.title(f"regression test set, $R^2$ = {r2:.3f}")
plt.tight_layout(); plt.show()

The network explains well over 90% of the variance of a target it has never seen an equation for (about 85% on the shorter `FAST_RUN` schedule). The
noise term, whose variance ($0.01$ in original units, roughly 1% of the target's variance) no model can
explain, sets the ceiling at $R^2 \approx 0.99$; whatever remains below that ceiling is approximation
error that a wider network or longer training would reduce.
Regression with an MLP differs from classification only in the last layer (one linear output instead of
$k$ logits) and the loss; everything else — layers, backward pass, optimizer — is shared. That modularity
is the point of Section 6.

### Exercise 6 (challenge): softmax backward without the combined trick

Implement a stand-alone `Softmax` layer whose `backward` uses the full Jacobian
$\partial p_i / \partial z_j = p_i (\delta_{ij} - p_j)$, and a separate `NLLLoss` that takes probabilities and
returns $-\frac{1}{N}\sum_n \log p_{n, y_n}$ with its own backward pass. Verify (a) that the gradient with
respect to the logits produced by the chain `Softmax` → `NLLLoss` equals the $(P - Y)/N$ of
`SoftmaxCrossEntropy`, and (b) that a network ending in `Softmax` passes `model_grad_check` with `NLLLoss`.

**Solution to Exercise 6**

Let $\mathbf{g} = \partial L / \partial \mathbf{p}$ be the upstream gradient for one example. The chain rule
through the softmax gives

$$
\frac{\partial L}{\partial z_j} = \sum_i g_i \frac{\partial p_i}{\partial z_j} = \sum_i g_i\, p_i (\delta_{ij} - p_j)
= p_j \Big( g_j - \sum_i g_i p_i \Big),
$$

so we never need to materialize the $k \times k$ Jacobian: a row-wise inner product $\sum_i g_i p_i$ and an
element-wise product suffice. For the NLL loss, $\partial L / \partial p_{n,j} = -\mathbb{1}[j = y_n] / (N p_{n, y_n})$.
Substituting $g_j = -\mathbb{1}[j = y]/(N p_y)$ into the softmax rule: $\sum_i g_i p_i = -1/N$, hence
$\partial L/\partial z_j = p_j(-\mathbb{1}[j=y]/(N p_y) + 1/N) = (p_j - \mathbb{1}[j = y]) / N$ — the combined
result, recovered the long way.

In [ ]:
class Softmax(Layer):
    def forward(self, z):
        self.P = softmax(z)
        return self.P

    def backward(self, grad_out):
        inner = np.sum(grad_out * self.P, axis=1, keepdims=True)    # sum_i g_i p_i, per row
        return self.P * (grad_out - inner)


class NLLLoss:
    """Negative log-likelihood of integer labels y given a matrix of probabilities P."""

    def forward(self, P, y):
        self.P, self.y, self.N = P, y, P.shape[0]
        return float(-np.mean(np.log(P[np.arange(self.N), y])))

    def backward(self):
        G = np.zeros_like(self.P)
        G[np.arange(self.N), self.y] = -1.0 / (self.P[np.arange(self.N), self.y] * self.N)
        return G


rng = np.random.default_rng(SEED)
Z_chk = rng.normal(size=(6, 5)); y_chk = rng.integers(0, 5, size=6)

# (a) the two routes to dL/dlogits agree
sm, nll = Softmax(), NLLLoss()
nll.forward(sm(Z_chk), y_chk)
grad_long_way = sm.backward(nll.backward())
combined = SoftmaxCrossEntropy(); combined.forward(Z_chk, y_chk)
grad_combined = combined.backward()
print("Softmax -> NLL gradient equals (P - Y)/N:", np.allclose(grad_long_way, grad_combined))

# (b) a full network ending in Softmax passes the gradient check with NLLLoss
net = Sequential(Linear(4, 8, rng, dtype=np.float64), ReLU(), Linear(8, 5, rng, dtype=np.float64), Softmax())
X_chk = rng.normal(size=(6, 4))
worst = model_grad_check(net, NLLLoss(), X_chk, y_chk, rng)
print(f"gradient check of the long way: worst relative error {worst:.2e} -> {'PASS' if worst < 1e-6 else 'FAIL'}")

Both checks pass. The combined `SoftmaxCrossEntropy` remains preferable in practice: it never forms
$\log p$ from an underflowed $p$, it is one line instead of two layers, and it is what
`F.cross_entropy` implements in PyTorch. But it is reassuring to see that the elegant $\mathbf{p} - \mathbf{y}$
is not a trick — it is what the chain rule produces.

## Summary

- An artificial neuron computes an affine function followed by a non-linearity; a single neuron is a linear classifier and cannot represent XOR.
- Hidden layers build new representations in which the classes become separable; stacking *linear* layers without activations collapses to a single linear map, so non-linear activations are essential.
- Sigmoid and tanh saturate (their derivatives vanish for large inputs); ReLU passes gradients unchanged when active but can die; leaky ReLU and GELU are common remedies.
- The forward pass of an MLP is a chain of matrix products with shapes $(N, d_{\text{in}})(d_{\text{in}}, d_{\text{out}})$; an MLP with sizes $d_0, \dots, d_L$ has $\sum_l (d_{l-1} + 1) d_l$ parameters.
- We train with a smooth loss — MSE for regression, softmax cross-entropy (computed via log-sum-exp) for classification — because accuracy is piecewise constant and has no useful gradient.
- Backpropagation applies the chain rule backwards through the graph. For a linear layer, $G_W = X^\top G_Z$, $G_{\mathbf{b}} = \sum_n (G_Z)_n$, $G_X = G_Z W^\top$; for element-wise activations $G_Z = G_A \odot \phi'(Z)$; for softmax cross-entropy $G_Z = (P - Y)/N$.
- Every hand-written gradient must be checked against central finite differences in float64; the relative error should be below $10^{-6}$.
- A layer abstraction (`forward`, `backward`, `params`, `grads`) plus a `Sequential` container and an `SGD` optimizer gives a small but complete deep learning library; in-place parameter updates are essential.
- With this library a 784-256-128-10 MLP reaches roughly 97–98% on MNIST; learning rate, batch size and initialization each change the outcome in predictable ways, and zero initialization fails because of symmetry.
- One hidden layer is a universal approximator in principle, but depth is far more parameter-efficient, and hand-derived backward passes do not scale to modern architectures — which is why the next notebook introduces automatic differentiation.

## Further reading

- Rumelhart, D. E., Hinton, G. E. and Williams, R. J., 1986. *Learning representations by back-propagating errors*. Nature 323, 533–536. The paper that made backpropagation standard.
- Rosenblatt, F., 1958. *The perceptron: a probabilistic model for information storage and organization in the brain*. Psychological Review 65(6). The original perceptron.
- Minsky, M. and Papert, S., 1969. *Perceptrons*. MIT Press. The book that formalised the limits of single-layer networks, XOR included.
- Cybenko, G., 1989. *Approximation by superpositions of a sigmoidal function*; Hornik, K., 1991. *Approximation capabilities of multilayer feedforward networks*; Leshno, M., Lin, V. Y., Pinkus, A. and Schocken, S., 1993. *Multilayer feedforward networks with a nonpolynomial activation function can approximate any function*. The universal approximation theorems.
- Montúfar, G., Pascanu, R., Cho, K. and Bengio, Y., 2014. *On the number of linear regions of deep neural networks*; Telgarsky, M., 2016. *Benefits of depth in neural networks*. Why depth beats width.
- He, K., Zhang, X., Ren, S. and Sun, J., 2015. *Delving deep into rectifiers: surpassing human-level performance on ImageNet classification*. Introduces the He initialization used here.
- Hendrycks, D. and Gimpel, K., 2016. *Gaussian Error Linear Units (GELUs)*.
- LeCun, Y., Bottou, L., Bengio, Y. and Haffner, P., 1998. *Gradient-based learning applied to document recognition*. Introduces MNIST and much of the modern training methodology.
- Goodfellow, I., Bengio, Y. and Courville, A., 2016. *Deep Learning*. MIT Press, Chapter 6 (Deep Feedforward Networks). https://www.deeplearningbook.org/
- Nielsen, M., 2015. *Neural Networks and Deep Learning*, Chapters 1–2 — a gentle, code-driven treatment of backpropagation. http://neuralnetworksanddeeplearning.com/
- Zhang, A., Lipton, Z. C., Li, M. and Smola, A. J. *Dive into Deep Learning*, Chapter 5 (Multilayer Perceptrons). https://d2l.ai/